# SafeViChi v2 · Hier-BidirLM-0.6B GuardMerge

[![Architecture](https://img.shields.io/badge/Nhánh_1-Hier--BidirLM--0.6B_GuardMerge-0369a1)](#sec-4)
[![Task](https://img.shields.io/badge/Tác_vụ-5_Nhãn_+_Bằng_chứng-0f766e)](#sec-4)
[![Dataset Commit](https://img.shields.io/badge/Data_Commit-371a0f1c-475569)](#sec-2)
[![Test Lock](https://img.shields.io/badge/Test_Suite-Locked_(1.670_mẫu)-b91c1c)](#sec-6)
[![License: MIT](https://img.shields.io/badge/License-MIT-15803d)](../LICENSE)

> **Tóm tắt:** Notebook huấn luyện, hiệu chuẩn và đánh giá nhánh kiến trúc phân cấp hai chiều **Hier-BidirLM-0.6B GuardMerge** (trộn trọng số `BidirLM-0.6B-Base` + `Qwen3Guard-Gen-0.6B`, tinh chỉnh LoRA + đầu phân cấp theo lượt FP32) trả về **5 nhãn tình huống ưu tiên** kèm **cụm từ bằng chứng chép nguyên văn (verbatim)**. Quy trình phân hoạch (`split_ids.json`), hiệu chuẩn, đánh giá song song `#5` (thô) / `#6` (+ chuẩn hóa giữ dòng) và khóa cứng bộ test (`1.670` mẫu) được thiết kế đồng nhất với Nhánh 2 ([`v2_dualorc_crf_colab.ipynb`](v2_dualorc_crf_colab.ipynb)) tại commit dữ liệu cố định `371a0f1caf3f7b62f62b87dbf557c66eda9c544a`.

---

### Hồ sơ thí nghiệm (Experiment Specification)

| Hạng mục | Thông số kỹ thuật |
| :--- | :--- |
| **Kiến trúc** | `Hier-BidirLM-0.6B GuardMerge` (Trộn trọng số BidirLM/Qwen3Guard + LoRA + `GuardHierarchy`) |
| **Backbone ghim revision** | `BidirLM/BidirLM-0.6B-Base` (`a6c88fa4`) + `Qwen/Qwen3Guard-Gen-0.6B` (`fada3b2f`), tỷ lệ trộn `MERGE_RATIO = 0.5` |
| **Nguồn dữ liệu khóa (`DATA_REV`)** | Commit `371a0f1caf3f7b62f62b87dbf557c66eda9c544a` (`so_sanh.jsonl`: 13.392 đoạn hội thoại `C0`, 1–8 lượt) |
| **Bộ Test đánh giá (Khóa cứng)** | `test_day_du.jsonl` (`334 C0`) + `test_perturb_C1.jsonl` (`668 C1`) + `test_perturb_C2.jsonl` (`668 C2`) = **`1.670` mẫu** |
| **Chế độ suy luận song song** | **Dòng `#5`** (đầu vào thô) & **Dòng `#6`** (chuẩn hóa 5 tầng giữ dòng `\n`, ánh xạ ngược span `non_verbatim == 0`) |
| **Tính tái hiện (Reproducibility)** | `SEED = 42`, `split_seed = 42`, ghim phiên bản thư viện, cơ chế khóa 1 lần `REAL_TEST_LOCK.json` & xác thực SHA-256 `calibration.json` |
| **Tài liệu gốc mô hình** | [Nghiên cứu BidirLM (mục 6.1)](https://arxiv.org/html/2604.02045#S6.SS1) · [Checkpoint BidirLM](https://huggingface.co/BidirLM/BidirLM-0.6B-Base) · [Checkpoint Qwen3Guard](https://huggingface.co/Qwen/Qwen3Guard-Gen-0.6B) |

---

### Mục lục (Table of Contents)

| Mục | Nội dung | Tệp xuất chính |
| :---: | :--- | :--- |
| **1** | [Thiết lập môi trường, Kiểm tra GPU & Bảng điều khiển Cấu hình](#sec-1) | `config.json`, `environment.json` |
| **2** | [Nạp dữ liệu ghim SHA-256 & Phân hoạch sạch không rò rỉ](#sec-2) | `split_ids.json` |
| **3** | [Tiền xử lý hội thoại, Tăng cường C1 phi điểm mù & Gom Bucket](#sec-3) | Tự kiểm định Offset / Aug |
| **4** | [Kiến trúc Backbone trộn trọng số & Đầu phân cấp `GuardMerge`](#sec-4) | Kiểm định Hai chiều & Padding |
| **5** | [Huấn luyện mô hình, Dual-View JS Consistency & Chọn Checkpoint](#sec-5) | `adapter.safetensors`, `last.pt`, `history.json` |
| **6** | [Hiệu chuẩn xác suất, Đánh giá phát triển & Cổng khóa Bộ Test](#sec-6) | `calibration.json`, `dev_metrics.json`, `REAL_TEST_LOCK.json`, `real_test_*` |
| **7** | [Kiểm thử độ bền (Stress Tests), Ngữ cảnh đa lượt & Benchmark](#sec-7) | `*_stress_tests.json`, `*_context_checks.json`, `benchmark.json` |
| **8** | [Lượng tử hóa INT8 trên CPU (bao gồm `embed_tokens`)](#sec-8) | `int8_comparison.json` |
| **9** | [Quy trình thực nghiệm chuẩn, Ablation & Đánh giá Test khóa](#sec-9) | Lệnh `python -m src.v2.eval.evaluate` |
| **10** | [Đối chiếu API & Điều kiện kiểm chứng thực nghiệm](#sec-10) | Bảng tham chiếu tài liệu kỹ thuật |
| **11** | [Phụ lục: Tương thích thư viện & Đặc tả quy ước đồng bộ](#sec-11) | Bộ thư viện ghim & quy ước đánh giá |

---

### Sơ đồ luồng thực nghiệm (End-to-End Pipeline)

```text
  [1. Môi trường & Cấu hình] ──▶ [2. Dữ liệu SHA-256 & Split sạch] ──▶ [3. Tokenizer + C1 Aug + Bucket]
                                                                                  │
  [6. Hiệu chuẩn (cal) & Khóa Test] ◀── [5. Train: LoRA + JS + Dice] ◀── [4. GuardMerge + Verify]
               │
               ├──▶ [7. Stress (5 bộ + #6), Context & Benchmark VRAM/Latency]
               └──▶ [8. Gộp LoRA & Nén INT8 CPU (PerRowInt8Embedding + Linear)]
```

### Chiến lược phân hoạch dữ liệu & Thiết kế kỹ thuật cốt lõi

| Chế độ (`DATA_SCHEME`) | `fit` | `dev` | `cal` | Mục đích sử dụng |
| :--- | :---: | :---: | :---: | :--- |
| **`dev_80_10_10`** *(mặc định)* | 10.713 (80%) | 1.340 (10%) | 1.339 (10%) | So sánh ứng viên song song & ablation; khớp 100% ID với Nhánh 2. |
| **`final_90_10`** | 12.053 (90%) | — | 1.339 (10%) | Huấn luyện toàn phần đủ `EPOCHS` cố định (không dùng `cal` để chọn epoch). |
| **`final_100_human_cal`** | 13.392 (100%) | — | `HUMAN_CAL_PATH` | Huấn luyện trên toàn bộ `so_sanh.jsonl` khi có tập validation người gán độc lập (khóa chặn 6 tệp train/test/C1/C2). |

- **Tăng cường phi điểm mù & Dual-View JS Consistency:** Ghép cặp song đôi `(bản gốc, bản bỏ dấu/C1)` trong cùng microbatch kèm `CONSISTENCY_WEIGHT = 0,25`, `ACCENT_AUG = 0,30`, `SURFACE_AUG = 0,20` (từ `>= 3` ký tự, ngân sách theo độ dài dòng) và `LABEL_SMOOTHING = 0,02`. Giữ **hoàn toàn ngoài train** 6 kiểu điểm mù `C2` (`blindspot.py`), `go_sai`, `dinh_tu`, `5/9/8`, `·` và teencode/phương ngữ held-out.
- **Chọn checkpoint & Hiệu chuẩn sát phân phối thật:** `CKPT_SELECT_MODE = "real_aligned"` sử dụng công thức tổng hợp `0.50 * macro_f1 + 0.15 * that_macro_f1 + 0.25 * that_acc + 0.10 * ev_f1`; `CAL_WEIGHT_MODE = "real_aligned"` áp dụng trần trọng số `cap = 3.0` và biên `T >= 0.74` trên `internal_llm`; hậu xử lý bằng chứng chống phình cụm + fallback có ngưỡng + **F1 theo ý** ([`DO_DO_BANG_CHUNG.md`](../docs/v2/DO_DO_BANG_CHUNG.md)).
- **Đo VRAM sạch & Nén INT8 toàn diện:** Giải phóng toàn bộ `opt`/`scheduler`/`scaler` trước `benchmark()`, nén 4× bảng `embed_tokens` (`593,5 MiB → 148,7 MiB`) qua `PerRowInt8Embedding`.
- **Khóa cứng Bộ Test đánh giá (`334 C0 + 668 C1 + 668 C2 = 1.670` mẫu, `#5` & `#6`):** Mặc định `EVALUATE_REAL_TEST = False`; mở khóa đúng 1 lần duy nhất để xuất `real_test_all_predictions.jsonl` (đủ `probs`, tương thích `python -m src.v2.eval.evaluate`), ma trận 15 lát cắt `real_test_official_matrix.json`, Dòng `#6` và khóa `REAL_TEST_LOCK.json`.

---

<a id="sec-1"></a>
## 1. Thiết lập môi trường, Kiểm tra GPU & Bảng điều khiển Cấu hình

### 1.1. Cài đặt các gói phụ thuộc đã ghim phiên bản

Chạy ô dưới đây để cài đúng bộ thư viện ghim (`transformers==4.57.6`, `tokenizers==0.22.2`, `safetensors==0.6.2`, `huggingface-hub==0.36.0`, `scikit-learn==1.9.1`). Nếu môi trường vừa thay đổi gói có sẵn, chọn **Runtime → Restart session** trước khi chạy tiếp.

In [ ]:
%pip -q install "transformers==4.57.6" "tokenizers==0.22.2" "safetensors==0.6.2" "huggingface-hub==0.36.0" "scikit-learn==1.9.1"

### 1.2. Kiểm tra phần cứng GPU & Bảng điều khiển Cấu hình Thí nghiệm

Hai ô code tiếp theo thực hiện tuần tự:
1. **Ô 1.2a (Kiểm tra phần cứng):** Xác nhận GPU CUDA, VRAM và ổ đĩa tạm (`/content` tối thiểu `5 GiB` vì mô hình tải hai checkpoint 0.6B và biến trung gian). Nếu dừng do thiếu GPU, vào **Runtime → Change runtime type → GPU** rồi chạy lại. Dung lượng hiển thị là ổ tạm runtime, không phải Google Drive.
2. **Ô 1.2b (Bảng điều khiển cấu hình & Khởi tạo phiên):** Nạp thư viện, khai báo tham số thí nghiệm & ablation, kiểm tra phiên bản gói ghim, cố định `SEED = 42`, gắn Google Drive và sinh thư mục `RUN` theo mã băm SHA-256 của `config.json`.

| Tham số trọng tâm | Mặc định | Ý nghĩa & Lưu ý an toàn |
| :--- | :---: | :--- |
| `RUN_MODE` | `"smoke"` | `"smoke"` kiểm thử nhanh 1 epoch; đổi `"full"` để huấn luyện đầy đủ 4 epoch. |
| `DATA_SCHEME` | `"dev_80_10_10"` | `"dev_80_10_10"` (so sánh ứng viên) · `"final_90_10"` · `"final_100_human_cal"`. |
| `MERGE_RATIO` / `TURN_MODE` / `HEAD_ARCH` | `0.5` / `"hier"` / `"v4"` | Tham số kiến trúc cốt lõi phục vụ ablation (`0.0`, `"flat"`, `"v3"`). |
| `EVALUATE_REAL_TEST` | `False` | **Khóa cứng** bộ test 1.670 mẫu; không nằm trong `RUN` hash để mở khóa **đúng 1 lần duy nhất** trên cùng checkpoint đã đóng băng `calibration.json`. |
| `EVAL_WITH_NORMALIZER` | `True` | Chấm song song **Dòng `#5` (thô)** và **Dòng `#6` (+ chuẩn hóa 5 tầng giữ dòng `\n`)**. |

In [ ]:
#@title 1.2a. Kiểm tra GPU CUDA, VRAM & Dung lượng đĩa tạm

from pathlib import Path
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU CUDA. Chọn Runtime → Change runtime type → GPU, sau đó chạy lại.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
free_gib = shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 2**30
print(f"GPU: {gpu_name} | VRAM: {gpu_gib:.1f} GiB | Ổ tạm còn trống: {free_gib:.1f} GiB")
if free_gib < 5.0:
    raise RuntimeError("Ổ tạm còn dưới 5 GiB. Xóa bớt tệp trước khi tải trọng số.")
if gpu_gib < 12.0:
    print("CẢNH BÁO: VRAM dưới 12 GiB. Có thể cần giảm BATCH hoặc CHUNK để tránh CUDA OOM.")

In [ ]:
#@title 1.2b. Bảng điều khiển Cấu hình Thí nghiệm & Khởi tạo Phiên (RUN Hash)
# ==============================================================================
# 1. NHẬP THƯ VIỆN & GÓI PHỤ THUỘC
# ==============================================================================
import os, sys, gc, json, hashlib, random, math, time, platform, unicodedata, re
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext
from itertools import islice
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from safetensors.torch import save_file, load_file
from tokenizers import Tokenizer
from transformers import Qwen3Config, Qwen3Model
import transformers, requests, sklearn, scipy, tokenizers, huggingface_hub, safetensors


# ==============================================================================
# 2. CẤU HÌNH THAM SỐ THÍ NGHIỆM
# ==============================================================================
#@markdown **Chế độ chạy & Chiến lược dữ liệu**
RUN_MODE = "smoke" #@param ["smoke", "full"]  # smoke = kiểm thử 1 epoch; đổi thành "full" để huấn luyện thật
USE_DRIVE = True #@param {type:"boolean"}
DATA_SCHEME = "dev_80_10_10" #@param ["dev_80_10_10", "final_90_10", "final_100_human_cal", "legacy_70_10_10_10"]  # 80/10/10 | 90/10 | 100% + human_cal | legacy
CKPT_SELECT_MODE = "real_aligned" #@param ["real_aligned", "macro_only"]  # "real_aligned" (0.50 macro + 0.15 that_f1 + 0.25 that_acc + 0.10 ev) | "macro_only"
CAL_WEIGHT_MODE = "real_aligned" #@param ["real_aligned", "uniform"]  # "real_aligned" (T: 70% thật + 30% sinh khó; bằng chứng: cân bằng 5 lớp) | "uniform"
HUMAN_CAL_PATH = None              # Đường dẫn JSONL validation người gán độc lập (tách khỏi train & test); bắt buộc nếu DATA_SCHEME="final_100_human_cal"
QUANTIZE_EMBEDDINGS = True #@param {type:"boolean"}  # Nén cả embed_tokens (INT8 theo hàng) trong ô INT8 CPU (594 -> 149 MiB)

#@markdown **Khóa cứng Bộ Test & Chấm song song #5/#6 (KHÔNG đưa vào RUN hash)**
# Khóa cứng tập test thật (KHÔNG đưa vào config hash để mở khóa 1 lần trên đúng RUN đã train!)
EVALUATE_AUDIT = False #@param {type:"boolean"}  # Chỉ dùng nếu DATA_SCHEME == "legacy_70_10_10_10"
EVALUATE_REAL_TEST = False #@param {type:"boolean"}  # KHÓA CỨNG: chỉ đặt True ĐÚNG 1 LẦN DUY NHẤT sau khi hoàn tất chọn cấu hình trên dev và đóng băng calibration.json
RUN_REAL_TEST_PERTURB = True #@param {type:"boolean"}  # Khi mở khóa real_test, chấm trọn bộ test_perturb_C1 (668) + test_perturb_C2 (668) và 5 bộ stress
EVAL_WITH_NORMALIZER = True #@param {type:"boolean"}  # Tự động chấm thêm Dòng #6 (#5 + chuẩn hóa 5 tầng giữ dòng, ánh xạ ngược span verbatim; KHÔNG đưa vào hash)


#@markdown **Kiến trúc GuardMerge, Hàm mất mát & Ablation (Yêu cầu #3)**
MERGE_RATIO = 0.5        # 0.0 = baseline BidirLM
TURN_MODE = "hier" #@param ["hier", "flat"]  # "flat" để đo đóng góp của phân cấp
HEAD_ARCH = "v4" #@param ["v4", "v3"]  # "v4" (đầy đủ) hoặc "v3" (biến thể tối giản cho ablation)
LORA_MLP = True #@param {type:"boolean"}  # học thêm gate_proj/up_proj/down_proj trong SwiGLU MLP
SPAN_WEIGHT = 0.35       # 0.0 để đo đóng góp của học bằng chứng
SPAN_DICE_WEIGHT = 0.5   # trộn BCE và Soft Dice ở đầu bằng chứng (0.0 để chỉ dùng BCE)
POS_WEIGHT_CAP = 3.0     # trần pos_weight sau khi lấy căn bậc hai tỷ lệ âm/dương
EVIDENCE_POOL = True #@param {type:"boolean"}  # nối tín hiệu bằng chứng về bộ phân loại
PROTOTYPE_WEIGHT = 0.0   # thử 0.1 trên dev với nguyên mẫu đơn nhãn
STRATUM_WEIGHT = True    # chỉ tăng nhẹ trọng số nhóm thật hiếm (QUAY_ROI|that, DE_DOA|that)
REAL_RARE_CAP = 2.5      # trần tối đa sau chuẩn hóa cho trọng số mẫu hiếm thật
LABEL_SMOOTHING = 0.02   # làm mượt nhẹ đồng bộ Nhánh 2, giảm FPR trên AN_TOAN
CONSISTENCY_WEIGHT = 0.25 # Dual-View JS Consistency trên cặp (gốc, bỏ dấu/C1); 0.0 để ablation
ACCENT_AUG = 0.30        # tỷ lệ cặp ghép bỏ dấu mỗi epoch (0.0 để ablation)
SURFACE_AUG = 0.20       # tỷ lệ cặp ghép biến dạng C1 đơn nghĩa phi điểm mù mỗi epoch (0.0 để ablation)
WORD_SCORE = "max" #@param ["max", "mean"]  # cách gộp điểm subword về mức từ ("max" hoặc "mean" cho ablation)
SPAN_POSTPROC = True #@param {type:"boolean"}  # tinh chỉnh thêm ngưỡng 1-từ, nối khoảng trống và fallback trên cal
SEED = 42
EPOCHS = 4 if RUN_MODE == "full" else 1

# ==============================================================================
# 3. KIỂM TRA PHẦN CỨNG, RÀNG BUỘC CẤU HÌNH & THƯ VIỆN GHIM
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Notebook cần GPU. Chọn Runtime → Change runtime type → GPU rồi chạy lại.")
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
GRAD_CKPT = gpu_gib < 12.0                             # False trên T4 (14.6 GiB) để nhanh hơn ~30-35%
BATCH, ACCUM = (16, 1) if gpu_gib >= 12.0 else (4, 4)  # batch hiệu dụng giữ nguyên = 16
EVAL_BATCH = 16
MAX_MICROBATCH_TOKENS = 2560 if not GRAD_CKPT else 6144
CHUNK, WIDTH, RANK = 192, 192, 16
LR_ADAPTER, LR_HEAD = 1e-4, 4e-4
RUN_INT8_CPU = False #@param {type:"boolean"}  # benchmark tùy chọn, cần thêm RAM CPU

assert DATA_SCHEME in {"dev_80_10_10", "final_90_10", "final_100_human_cal", "legacy_70_10_10_10"}
if DATA_SCHEME == "final_100_human_cal" and not HUMAN_CAL_PATH:
    raise ValueError("DATA_SCHEME='final_100_human_cal' bắt buộc phải đặt HUMAN_CAL_PATH (validation người gán độc lập) để không rò rỉ hiệu chuẩn!")
assert CKPT_SELECT_MODE in {"real_aligned", "macro_only"}
assert CAL_WEIGHT_MODE in {"real_aligned", "uniform"}
assert 0 <= ACCENT_AUG <= 1 and 0 <= SURFACE_AUG <= 1 and (ACCENT_AUG + SURFACE_AUG) <= 1.0 and 0 <= SPAN_DICE_WEIGHT <= 1
assert CONSISTENCY_WEIGHT >= 0
assert 0 <= LABEL_SMOOTHING < 1 and POS_WEIGHT_CAP >= 1.0 and REAL_RARE_CAP >= 1.0
assert PROTOTYPE_WEIGHT >= 0 and SPAN_WEIGHT >= 0
assert RUN_MODE in {"full", "smoke"} and TURN_MODE in {"hier", "flat"} and 0 <= MERGE_RATIO <= 1
assert HEAD_ARCH in {"v3", "v4"} and WORD_SCORE in {"max", "mean"}
assert BATCH >= 1 and ACCUM >= 1 and EVAL_BATCH >= 1 and MAX_MICROBATCH_TOKENS >= CHUNK + 1

native_bf16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)
if native_bf16:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
DTYPE = (torch.bfloat16 if native_bf16 else torch.float16) if DEVICE.type == "cuda" else torch.float32

pinned = {
    "transformers": (transformers.__version__, "4.57.6"),
    "tokenizers": (tokenizers.__version__, "0.22.2"),
    "huggingface-hub": (huggingface_hub.__version__, "0.36.0"),
    "safetensors": (safetensors.__version__, "0.6.2"),
    "scikit-learn": (sklearn.__version__, "1.9.1"),
}
for package, (active, expected) in pinned.items():
    if active != expected:
        raise RuntimeError(f"{package} đang dùng {active}, cần {expected}. Chạy ô cài đặt, khởi động lại kernel rồi Run all.")


# ==============================================================================
# 4. CỐ ĐỊNH SEED, GẮN DRIVE & KHỞI TẠO THƯ MỤC RUN (CONFIG / ENV)
# ==============================================================================
def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
seed_all(SEED)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/SafeViChi/guardmerge") if USE_DRIVE else Path.cwd() / "guardmerge_runs"
DATA_REV = "371a0f1caf3f7b62f62b87dbf557c66eda9c544a"
BASE, BREV = "BidirLM/BidirLM-0.6B-Base", "a6c88fa4f7653ad82237ba00d85b1e68b8255316"
GUARD, GREV = "Qwen/Qwen3Guard-Gen-0.6B", "fada3b2f655b89601929198343c94cd2f64d93cc"
LABELS = ["AN_TOAN", "XUC_PHAM", "QUAY_ROI", "DE_DOA", "LUA_DAO"]
config = dict(
    schema=6,
    data_scheme=DATA_SCHEME,
    ckpt_select_mode=CKPT_SELECT_MODE,
    ckpt_select_weights=(([0.50, 0.15, 0.25, 0.10] if SPAN_WEIGHT > 0 else [0.55, 0.20, 0.25, 0.0]) if CKPT_SELECT_MODE == "real_aligned" else None),
    cal_weight_mode=CAL_WEIGHT_MODE,
    human_cal_sha256=(hashlib.sha256(Path(HUMAN_CAL_PATH).read_bytes().removeprefix(b"\xef\xbb\xbf").replace(b"\r\n", b"\n")).hexdigest() if HUMAN_CAL_PATH else None),
    head_arch=HEAD_ARCH,
    evidence_pool=EVIDENCE_POOL,
    prototype=PROTOTYPE_WEIGHT,
    consistency_weight=CONSISTENCY_WEIGHT,
    accent_aug=ACCENT_AUG,
    surface_aug=SURFACE_AUG,
    stratum_weight=STRATUM_WEIGHT,
    real_rare_cap=REAL_RARE_CAP,
    label_smoothing=LABEL_SMOOTHING,
    span_dice=SPAN_DICE_WEIGHT,
    pos_weight_cap=POS_WEIGHT_CAP,
    word_score=WORD_SCORE,
    span_postproc=SPAN_POSTPROC,
    lora_mlp=LORA_MLP,
    dataset=DATA_REV,
    base=BASE,
    base_rev=BREV,
    guard=GUARD,
    guard_rev=GREV,
    ratio=MERGE_RATIO,
    mode=RUN_MODE,
    turns=TURN_MODE,
    span_weight=SPAN_WEIGHT,
    seed=SEED,
    split_seed=42,
    epochs=EPOCHS,
    batch=BATCH,
    accum=ACCUM,
    max_microbatch_tokens=MAX_MICROBATCH_TOKENS,
    grad_ckpt=GRAD_CKPT,
    chunk=CHUNK,
    width=WIDTH,
    rank=RANK,
    lr_adapter=LR_ADAPTER,
    lr_head=LR_HEAD,
    labels=LABELS,
    dtype=str(DTYPE),
)
def digest(b): return hashlib.sha256(b).hexdigest()
RUN = ROOT / digest(json.dumps(config, sort_keys=True).encode())[:14]
RUN.mkdir(parents=True, exist_ok=True)
def write_json(path, obj):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf8")
    os.replace(tmp, path)
write_json(RUN / "config.json", config)
environment = dict(
    python=sys.version,
    torch=str(torch.__version__),
    cuda=torch.version.cuda,
    device=torch.cuda.get_device_name() if DEVICE.type == "cuda" else platform.processor(),
    capability=torch.cuda.get_device_capability() if DEVICE.type == "cuda" else None,
    gpu_gib=round(gpu_gib, 2),
    native_bf16=native_bf16,
    dtype=str(DTYPE),
    fp16_scaler=DTYPE == torch.float16,
    data_scheme=DATA_SCHEME,
    numpy=np.__version__,
    scipy=scipy.__version__,
    pandas=pd.__version__,
    requests=requests.__version__,
    **{name: active for name, (active, _) in pinned.items()},
)
write_json(RUN / "environment.json", environment)
print(RUN, f"HEAD={HEAD_ARCH} SCHEME={DATA_SCHEME} BATCH={BATCH}x{ACCUM}", environment)

---
<a id="sec-2"></a>
## 2. Nạp dữ liệu ghim SHA-256 & Phân hoạch sạch không rò rỉ

Ô này tải dữ liệu sạch tại commit cố định `371a0f1caf3f7b62f62b87dbf557c66eda9c544a`, đối chiếu **SHA-256** của `so_sanh.jsonl` và `train.jsonl` (`13.392` bản ghi `C0`), đồng thời xác thực nhãn, nguồn, số lượt hội thoại và cụm bằng chứng verbatim.

| Bảo đảm tính toàn vẹn | Cơ chế áp dụng trong ô 2.1 |
| :--- | :--- |
| **Khớp 100% phân hoạch liên nhánh** | `StratifiedGroupKFold(n_splits=10, random_state=42)` giữ nguyên 100% ID `dev` (1.340) và `cal` (1.339) với Nhánh 2 ([`v2_dualorc_crf_colab.ipynb`](v2_dualorc_crf_colab.ipynb)). |
| **Cách ly tuyệt đối Bộ Test đánh giá** | Cả 3 tệp `test_day_du.jsonl` (334 C0), `test_perturb_C1.jsonl` (668 C1) và `test_perturb_C2.jsonl` (668 C2) **không được nạp ở ô này** — chỉ đọc ở Mục 6 sau khi đóng băng checkpoint và `calibration.json`. |
| **Chống rò rỉ `HUMAN_CAL_PATH`** | Kiểm tra SHA-256 chặn ngay nếu trỏ nhầm vào bất kỳ tệp nào trong 6 tệp `train`/`test`/`C1`/`C2` đã ghim (kể cả bản đổi tên, CRLF hoặc UTF-8 BOM). |

In [ ]:
#@title 2.1. Tải dữ liệu ghim SHA-256, Kiểm tra toàn vẹn & Chia tập StratifiedGroupKFold
# ==============================================================================
# 1. PHIÊN HTTP RETRY & BẢNG BĂM SHA-256 GHIM
# ==============================================================================
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_HTTP = requests.Session()
DATA_HTTP.mount("https://", HTTPAdapter(max_retries=Retry(total=3, connect=3, read=3,
    backoff_factor=1, status_forcelist=(429, 500, 502, 503, 504))))
DATA_URL = f"https://raw.githubusercontent.com/thaoanhlehoang/SafeViChi/{DATA_REV}/data/v2/"
DATA_SHA256 = {
    "so_sanh.jsonl": "c2c5046fcd2e3853e76ecc79c7ea70eaad79743380e03185bc32610ebc813102",
    "train.jsonl": "4a31dcee5b93392f1b5d6fba778d9121604aa59beaa8252b3026ce1112ca334f",
    "test.jsonl": "9ebdad75d0b1bc60df7d1cacd761dd74c5eb6e64d678692f4d2889204033c115",
    "test_day_du.jsonl": "7652e102a6e8b6b9ce9c99633333afe639642e65f41f2d6edc005a8e75085463",
    "test_perturb_C1.jsonl": "1ae3cadecac4df407273d23a87c84201f603311a4994acaaebfd3915736585a5",
    "test_perturb_C2.jsonl": "8b2a24234dda9e855c91d86ae03d21a16710be2dc2569d121d29d5b7f693823b",
}
TEENCODE_DICT_SHA256 = "c7f345574974668c2b319955c23df11b5d6ac3055e743cb1b4e40ec326ff1c14"

def _matches_blob_sha(raw_bytes, expected):
    clean = raw_bytes.removeprefix(b"\xef\xbb\xbf")
    return digest(raw_bytes) == expected or digest(clean) == expected or digest(clean.replace(b"\r\n", b"\n")) == expected

LOCKED_DATA_FILES = (
    "test.jsonl", "test_day_du.jsonl", "test_perturb_C1.jsonl", "test_perturb_C2.jsonl",
    "so_sanh.jsonl", "train.jsonl",
)

# ==============================================================================
# 2. KHÓA AN TOÀN HUMAN_CAL_PATH & HÀM TẢI TỆP DỮ LIỆU
# ==============================================================================
if HUMAN_CAL_PATH:
    _hcal_raw = Path(HUMAN_CAL_PATH).read_bytes()
    for _locked_name in LOCKED_DATA_FILES:
        if _matches_blob_sha(_hcal_raw, DATA_SHA256[_locked_name]):
            raise ValueError(f"KHÓA AN TOÀN: HUMAN_CAL_PATH trùng SHA-256 của {_locked_name} — từ chối để chống rò rỉ!")
    del _hcal_raw, _locked_name

def fetch_teencode_dict(expected=TEENCODE_DICT_SHA256):
    """Tải từ điển teencode v1 (643 mục) đã ghim SHA-256 phục vụ đánh giá Dòng #6 (chuẩn hóa đầu vào)."""
    local_candidates = (
        Path.cwd() / "src" / "shared" / "normalization" / "teencode_dict.json",
        Path.cwd().parent / "src" / "shared" / "normalization" / "teencode_dict.json",
        Path.cwd() / "src" / "normalization" / "teencode_dict.json",
        Path.cwd().parent / "src" / "normalization" / "teencode_dict.json",
    )
    for cand in local_candidates:
        if cand.exists() and _matches_blob_sha(cand.read_bytes(), expected):
            return json.loads(cand.read_text(encoding="utf-8-sig"))
    cache = Path.cwd() / ".safevichi_data" / DATA_REV
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / "teencode_dict.json"
    if not path.exists() or not _matches_blob_sha(path.read_bytes(), expected):
        base_raw = f"https://raw.githubusercontent.com/thaoanhlehoang/SafeViChi/{DATA_REV}/"
        last_err = None
        for rel in ("src/normalization/teencode_dict.json", "src/shared/normalization/teencode_dict.json"):
            try:
                res = DATA_HTTP.get(base_raw + rel, timeout=(15, 60))
                if res.status_code == 200 and _matches_blob_sha(res.content, expected):
                    tmp_p = path.with_suffix(f".{os.getpid()}.tmp")
                    tmp_p.write_bytes(res.content)
                    os.replace(tmp_p, path)
                    break
                last_err = f"HTTP {res.status_code} ({rel})"
            except Exception as exc:
                last_err = str(exc)
        else:
            raise RuntimeError(f"Không tải được teencode_dict.json khớp SHA-256: {last_err}")
    return json.loads(path.read_text(encoding="utf-8-sig"))

def fetch_data(name, expected=None):
    expected = expected or DATA_SHA256[name]
    for candidate in (Path.cwd() / "data" / "v2" / name, Path.cwd().parent / "data" / "v2" / name):
        if candidate.exists() and _matches_blob_sha(candidate.read_bytes(), expected):
            return [json.loads(line) for line in candidate.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    cache = Path.cwd() / ".safevichi_data" / DATA_REV
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / name
    if not path.exists() or not _matches_blob_sha(path.read_bytes(), expected):
        res = DATA_HTTP.get(DATA_URL + name, timeout=(15, 180))
        res.raise_for_status()
        assert _matches_blob_sha(res.content, expected), f"Checksum sai: {name}"
        tmp_p = path.with_suffix(f".{os.getpid()}.tmp")
        tmp_p.write_bytes(res.content)
        os.replace(tmp_p, path)
    return [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]


# ==============================================================================
# 3. XÁC THỰC TẬP HUẤN LUYỆN SẠCH (13.392 ĐOẠN C0)
# ==============================================================================
rows = fetch_data("so_sanh.jsonl")
train_rows = fetch_data("train.jsonl")
assert len(rows) == len(train_rows) == 13392
assert len({r["id"] for r in rows}) == len({r["nhom_trung"] for r in rows}) == len(rows)
for r, t in zip(rows, train_rows):
    assert t == {k: r[k] for k in ("id", "text", "nhan", "bang_chung")}
    assert r["nhan"] in LABELS and r["bien_dang"] == "C0"
    assert r["so_dong"] == len(r["text"].splitlines()) and 1 <= r["so_dong"] <= 8
    assert all(any(p in line for line in r["text"].splitlines()) for p in r["bang_chung"])
    assert (r.get("meta") or {}).get("kich_ban") not in {"LD7", "LD12", "DD6", "QR6"}, f"Rò rỉ kịch bản ngoài train: {r['id']}"
del train_rows


# ==============================================================================
# 4. PHÂN HOẠCH STRATIFIEDGROUPKFOLD & LƯU SPLIT_IDS.JSON
# ==============================================================================
def split_group(r):
    meta = r.get("meta") or {}
    if meta.get("loai") == "sinh":
        return "prompt/" + r["nguon_id"].rsplit(":", 1)[0].removesuffix("-SD")
    grp = r.get("nhom_trung")
    return "real/" + (str(grp) if grp else str(r["id"]))

def make_splits(all_rows, scheme=DATA_SCHEME, split_seed=42):
    groups = np.array([split_group(r) for r in all_rows])
    strata = [r["nhan"] + "/" + r["meta"]["loai"] for r in all_rows]
    fold = np.full(len(all_rows), -1)
    for i, (_, ix) in enumerate(StratifiedGroupKFold(10, shuffle=True, random_state=split_seed).split(groups, strata, groups)):
        fold[ix] = i
    if scheme == "dev_80_10_10":
        masks = {"fit": (fold >= 3) | (fold == 0), "dev": fold == 2, "cal": fold == 1}
    elif scheme == "final_90_10":
        masks = {"fit": fold != 1, "cal": fold == 1}
    elif scheme == "final_100_human_cal":
        masks = {"fit": np.ones(len(all_rows), dtype=bool)}
    elif scheme == "legacy_70_10_10_10":
        masks = {"fit": fold >= 3, "dev": fold == 2, "cal": fold == 1, "audit": fold == 0}
    else:
        raise ValueError(f"DATA_SCHEME không hợp lệ: {scheme}")
    out = {k: [all_rows[i] for i in np.flatnonzero(mask)] for k, mask in masks.items()}
    assert len(set.union(*(set(map(split_group, v)) for v in out.values()))) == sum(len(set(map(split_group, v))) for v in out.values())
    assert sum(len(v) for v in out.values()) == len(all_rows)
    return out

parts = make_splits(rows, scheme=DATA_SCHEME, split_seed=config["split_seed"])
if RUN_MODE == "smoke":
    for k, v in parts.items():
        per_label = 8 if k == "fit" else 2
        parts[k] = [r for label in LABELS for r in [x for x in v if x["nhan"] == label][:per_label]]
for v in parts.values():
    assert set(r["nhan"] for r in v) == set(LABELS)
write_json(RUN / "split_ids.json", {"scheme": DATA_SCHEME, **{k: [r["id"] for r in v] for k, v in parts.items()}})
print(f"DATA_SCHEME = {DATA_SCHEME} | Phân bổ số mẫu mỗi split:")
print(pd.DataFrame({k: Counter(r["nhan"] for r in v) for k, v in parts.items()}))
print(pd.crosstab([r["nhan"] for r in rows], [r["meta"]["loai"] for r in rows]))

---
<a id="sec-3"></a>
## 3. Tiền xử lý hội thoại, Tăng cường C1 phi điểm mù & Gom Bucket

Ô này thiết lập tokenizer, phép tăng cường dữ liệu giữ nguyên vị trí bằng chứng và cơ chế gom batch theo độ dài:

| Thành phần | Thiết kế kỹ thuật |
| :--- | :--- |
| **Bảo toàn chuỗi & Offset gốc** | Giữ nguyên chuỗi và tọa độ ký tự để trích bằng chứng verbatim về văn bản gốc. Lượt dài được cắt thành các khối liên tiếp (`CHUNK = 192`) để không rơi phần đuôi; cụm lặp nhiều vị trí được gán nhãn bỏ qua (`-100`) khi tính loss. |
| **Tăng cường C1 đơn nghĩa (trên `fit`)** | Chỉ dùng bỏ dấu (`augment_accents`) và tập con quy tắc C1 đơn nghĩa thuộc vùng phủ `normalizer.py` (`augment_surface`: 17 teencode đơn nghĩa, lookalike `a/e/i/o`, dấu tách `. - _` và kéo dài $\ge 3$ ký tự). |
| **Giữ hoàn toàn ngoài train (Held-out / C2)** | Loại trừ hoàn toàn đại từ đơn ký tự mơ hồ (`a/e/m/t`), phương ngữ/tiếng lóng (`dzô/dìa/fen/rứa/mô/hổng`), lookalike `s/g/b`, dấu tách `·`, lỗi gõ phím QWERTY và toàn bộ kỹ thuật điểm mù `blindspot.py`. |
| **Gom nhóm `BucketBatchSampler`** | Gom mẫu theo chiều dài và tách `split_microbatch` không cắt đôi cặp `(gốc, biến dạng)`, giảm padding mà vẫn bảo toàn đúng tập chỉ số mỗi epoch. |

In [ ]:
#@title 3.1. Token hóa, Tăng cường C1 phi điểm mù, Gom Bucket & Tự kiểm định
# ==============================================================================
# 1. TẢI TOKENIZER & CẤU HÌNH TOKEN ĐẶC BIỆT
# ==============================================================================
def hub(repo, name, rev):
    return hf_hub_download(repo, name, revision=rev)
tok_path = hub(BASE, "tokenizer.json", BREV)
tok = Tokenizer.from_file(tok_path)
tok.no_truncation()
tok.no_padding()
BOS, PAD = 151644, 151645


# ==============================================================================
# 2. CHUẨN HÓA BỎ DẤU & TĂNG CƯỜNG BỀ MẶT C1 PHI ĐIỂM MÙ
# ==============================================================================
def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn").translate(
        str.maketrans({"đ": "d", "Đ": "D"}))

# Tập quy tắc C1 đơn nghĩa dùng trong train (loại đại từ 1 ký tự mơ hồ a/e/m/t; giữ phương ngữ, slang, s/g/b, ·/~ ngoài train)
C1_TEENCODE_MAP = {
    "không": ("ko", "k", "hông"),
    "được": ("đc", "dc"),
    "biết": ("bt", "bít"),
    "với": ("vs",),
    "người": ("ng",),
    "mình": ("mk",),
    "gì": ("j",),
    "vậy": ("zậy",),
    "rồi": ("rùi",),
    "cũng": ("cx",),
    "phải": ("fải",),
    "nhiều": ("nhìu",),
    "yêu": ("iu",),
    "thôi": ("thui",),
    "hiểu": ("hỉu",),
    "quá": ("qá", "wá"),
    "chưa": ("chưaaa",),
}
C1_LOOKALIKE = {"a": ("4", "@"), "e": ("3",), "i": ("1",), "o": ("0",)}
C1_SEPS = (".", "-", "_")
_ALPHA_CORE_RE = re.compile(r"^[a-zA-ZÀ-ỹ]+$")

def _split_affix(token):
    i, j = 0, len(token)
    while i < j and not (token[i].isalnum() or unicodedata.combining(token[i])):
        i += 1
    while j > i and not (token[j - 1].isalnum() or unicodedata.combining(token[j - 1])):
        j -= 1
    return token[:i], token[i:j], token[j:]

def _map_pos(pos, segs, is_end):
    for sa, sb, oa, ob, same in segs:
        if (pos < sb) if not is_end else (pos <= sb):
            if pos <= sa:
                return oa
            if same:
                return oa + (pos - sa)
            return ob if is_end else oa
    return segs[-1][3] if segs else 0

def transform_line_with_spans(line, spans, core_fn):
    segs = []
    parts_out = []
    cur = 0
    for m in re.finditer(r"\S+|\s+", line):
        a, b = m.span()
        tok_str = m.group(0)
        if tok_str.isspace():
            parts_out.append(tok_str)
            segs.append((a, b, cur, cur + len(tok_str), True))
            cur += len(tok_str)
            continue
        lead, core, trail = _split_affix(tok_str)
        pos = a
        if lead:
            parts_out.append(lead)
            segs.append((pos, pos + len(lead), cur, cur + len(lead), True))
            cur += len(lead)
            pos += len(lead)
        if core:
            new_core = core_fn(core) or core
            parts_out.append(new_core)
            segs.append((pos, pos + len(core), cur, cur + len(new_core), new_core == core))
            cur += len(new_core)
            pos += len(core)
        if trail:
            parts_out.append(trail)
            segs.append((pos, pos + len(trail), cur, cur + len(trail), True))
            cur += len(trail)
    out_line = "".join(parts_out)
    new_spans = []
    for sa, sb in spans:
        na = _map_pos(sa, segs, False)
        nb = _map_pos(sb, segs, True)
        if na < nb:
            new_spans.append((na, nb))
    return out_line, new_spans

def _apply_line_transforms(row, line_fn_at):
    lines = row["text"].split("\n")
    unique_phrases = list(dict.fromkeys(row["bang_chung"]))
    new_lines = []
    new_phrases = []
    seen = set()
    for idx_line, line in enumerate(lines):
        line_spans = []
        for ph in unique_phrases:
            for m in re.finditer("(?=" + re.escape(ph) + ")", line):
                line_spans.append((m.start(), m.start() + len(ph)))
        out_line, mapped_spans = transform_line_with_spans(line, line_spans, line_fn_at(idx_line, line))
        new_lines.append(out_line)
        for na, nb in mapped_spans:
            sub = out_line[na:nb]
            if sub and sub not in seen:
                seen.add(sub)
                new_phrases.append(sub)
    return dict(row, text="\n".join(new_lines), bang_chung=new_phrases)

def augment_accents(row, rng):
    lines = row["text"].split("\n")
    accented_idx = [i for i, l in enumerate(lines) if strip_accents(l) != l]
    if not accented_idx:
        return row
    if len(lines) > 1 and len(accented_idx) > 1 and rng.random() < 0.40:
        k = rng.randint(1, len(accented_idx) - 1)
        chosen = set(rng.sample(accented_idx, k))
    else:
        chosen = set(accented_idx)
    return _apply_line_transforms(row, lambda idx, _: (strip_accents if idx in chosen else (lambda c: c)))

def _c1_lookalike_word(core, rng):
    if len(core) < 3 or not _ALPHA_CORE_RE.match(core):
        return core
    idxs = [i for i in range(1, len(core)) if core[i].lower() in C1_LOOKALIKE]
    if not idxs:
        return core
    max_k = max(1, (len(core) - 1) // 2)  # bảo đảm số chữ cái > số ký tự đổi được (khớp normalizer.py)
    picks = set(rng.sample(idxs, min(len(idxs), rng.choice((1, 2)), max_k)))
    return "".join(rng.choice(C1_LOOKALIKE[ch.lower()]) if i in picks else ch for i, ch in enumerate(core))

def _c1_sep_word(core, rng):
    if len(core) < 3 or not _ALPHA_CORE_RE.match(core):
        return core
    cuts = sorted(rng.sample(range(1, len(core)), min(len(core) - 1, rng.choice((1, 2)))))
    sep = rng.choice(C1_SEPS)
    out, prev = [], 0
    for c in cuts:
        out.append(core[prev:c])
        out.append(sep)
        prev = c
    out.append(core[prev:])
    return "".join(out)

def _c1_elongate_word(core, rng):
    if len(core) < 2 or not _ALPHA_CORE_RE.match(core):
        return core
    # Lặp >= 3 lần để khớp quy tắc B4 của normalizer (không dùng lặp đôi điểm mù)
    vowels = [i for i, c in enumerate(core) if c.lower() in "aeiouyăâêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"]
    pos = rng.choice(vowels) if vowels else len(core) - 1
    ch = core[pos]
    extra = rng.choice((2, 3))
    return core[:pos + 1] + (ch * extra) + core[pos + 1:]

def augment_surface(row, rng):
    do_lower = rng.random() < 0.50
    do_teencode = rng.random() < 0.75
    do_strip = rng.random() < 0.40
    obfuscate_mode = rng.choice(("teencode_strip", "lookalike", "sep", "elongate"))
    def make_fn(idx, line):
        n_w = max(1, len(line.split()))
        budget = [min(6, max(2, round(n_w * rng.uniform(0.25, 0.45)))) if obfuscate_mode != "teencode_strip" else 0]
        def fn(core):
            w = core
            if do_teencode:
                alts = C1_TEENCODE_MAP.get(w.lower())
                if alts and rng.random() < 0.45:
                    rep = rng.choice(alts)
                    w = rep.capitalize() if (w[0].isupper() and not do_lower) else rep
            if do_strip:
                w = strip_accents(w)
            if do_lower:
                w = w.lower()
            if budget[0] > 0 and len(w) >= 3 and _ALPHA_CORE_RE.match(w) and rng.random() < 0.55:
                if obfuscate_mode == "lookalike":
                    cand = _c1_lookalike_word(w, rng)
                elif obfuscate_mode == "sep":
                    cand = _c1_sep_word(w, rng)
                else:
                    cand = _c1_elongate_word(w, rng)
                if cand != w:
                    w = cand
                    budget[0] -= 1
            return w
        return fn
    return _apply_line_transforms(row, make_fn)


# ==============================================================================
# 3. KHỚP VỊ TRÍ BẰNG CHỨNG & ĐÓNG GÓI KHỐI HỘI THOẠI (PREPARE)
# ==============================================================================
def annotations(row):
    text = row["text"]
    yes, unknown = [], []
    for phrase in set(row["bang_chung"]):
        positions = [m.start() for m in re.finditer("(?=" + re.escape(phrase) + ")", text)]
        target = yes if len(positions) == 1 else unknown
        target.extend((s, s + len(phrase)) for s in positions)
    return yes, unknown

def overlaps(interval, spans):
    a, b = interval
    return any(a < end and start < b for start, end in spans)

def prepare(row):
    text = row["text"]
    yes, unknown = annotations(row)
    skip = row["nhan"] != "AN_TOAN" and not yes and not unknown
    lines = [text] if TURN_MODE == "flat" else text.split("\n")
    chunks = []
    cursor = 0
    for turn, line in enumerate(lines):
        seq = tok.encode(line, add_special_tokens=False)
        offsets = []
        for a, b in seq.offsets:
            # NFC của tokenizer có thể bỏ dấu tổ hợp ở cuối offset gốc.
            while a < b and b < len(line) and unicodedata.combining(line[b]):
                b += 1
            offsets.append((cursor + a, cursor + b))
        for start in range(0, len(seq.ids), CHUNK):
            off = offsets[start:start + CHUNK]
            targets = [
                -100 if skip or a == b or overlaps((a, b), unknown) else int(overlaps((a, b), yes))
                for a, b in off
            ]
            chunks.append((turn, seq.ids[start:start + CHUNK], off, targets))
        cursor += len(line) + 1
    if not chunks:
        raise ValueError(f"Văn bản rỗng {row['id']}")
    max_len = max(len(c[1]) for c in chunks) + 1
    return dict(row=row, chunks=chunks, label=LABELS.index(row["nhan"]), max_len=max_len, n_chunks=len(chunks))


# ==============================================================================
# 4. COLLATE, BUCKETBATCHSAMPLER & CHIA MICROBATCH GIỮ NGUYÊN CẶP
# ==============================================================================
def collate(items):
    """Nhận danh sách item: mỗi item là 1 dict prepare(row) hoặc tuple (clean, pert) cho Dual-View."""
    examples, pairs = [], []
    for it in items:
        if isinstance(it, tuple):
            i0 = len(examples)
            examples.append(it[0])
            examples.append(it[1])
            pairs.append((i0, i0 + 1))
        else:
            examples.append(it)
    chunks = [(i, *chunk) for i, e in enumerate(examples) for chunk in e["chunks"]]
    n = max(len(ids) for _, _, ids, _, _ in chunks) + 1
    tokens = torch.full((len(chunks), n), PAD, dtype=torch.long)
    mask = torch.zeros_like(tokens, dtype=torch.bool)
    targets = torch.full_like(tokens, -100)
    owners, turns, offsets = [], [], []
    for j, (i, t, ids, off, labels) in enumerate(chunks):
        end = len(ids) + 1
        tokens[j, :end] = torch.tensor([BOS] + ids)
        mask[j, :end] = True
        targets[j, 1:end] = torch.tensor(labels)
        owners.append(i)
        turns.append(t)
        offsets.append(off)
    return dict(
        ids=tokens,
        mask=mask,
        targets=targets,
        owner=torch.tensor(owners),
        turn=torch.tensor(turns),
        offsets=offsets,
        examples=examples,
        items=items,
        pairs=pairs,
        labels=torch.tensor([e["label"] for e in examples]),
    )

def _item_sort_key(it):
    if isinstance(it, tuple):
        return (max(it[0]["max_len"], it[1]["max_len"]), it[0]["n_chunks"] + it[1]["n_chunks"])
    return (it["max_len"], it["n_chunks"])

class BucketBatchSampler:
    def __init__(self, data, batch_size, seed=SEED, epoch=0, mega_mult=32):
        self.data = data
        self.batch_size = batch_size
        self.seed = seed
        self.epoch = epoch
        self.mega_mult = mega_mult
    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        idx = list(range(len(self.data)))
        rng.shuffle(idx)
        mega = self.batch_size * self.mega_mult
        batches_idx = []
        for s in range(0, len(idx), mega):
            pool = idx[s:s + mega]
            pool.sort(key=lambda i: _item_sort_key(self.data[i]))
            for j in range(0, len(pool), self.batch_size):
                batches_idx.append(pool[j:j + self.batch_size])
        rng.shuffle(batches_idx)
        return iter(batches_idx)
    def __len__(self):
        return math.ceil(len(self.data) / self.batch_size)

def split_microbatch(batch, max_tokens=MAX_MICROBATCH_TOKENS):
    items = batch["items"]
    if batch["ids"].numel() <= max_tokens or len(items) <= 1:
        return [batch]
    mid = len(items) // 2
    return (
        split_microbatch(collate(items[:mid]), max_tokens)
        + split_microbatch(collate(items[mid:]), max_tokens)
    )


# ==============================================================================
# 5. TỰ KIỂM ĐỊNH OFFSET / AUGMENTATION & MÃ HÓA DỮ LIỆU
# ==============================================================================
# Đối chiếu offset trên chuỗi gốc + kiểm thử tăng cường C1 giữ vị trí bằng chứng
for form in ("NFC", "NFD", "bo_dau"):
    needle = unicodedata.normalize("NFD" if form == "NFD" else "NFC", "dữ")
    if form == "bo_dau":
        needle = strip_accents(needle)
    sample = "Chào bạn!\n" + needle + " thoại"
    e = prepare(dict(id="offset-check", text=sample, nhan="XUC_PHAM", bang_chung=[needle]))
    start = sample.index(needle)
    marked = {
        i for _, _, offsets, ys in e["chunks"] for (a, b), y in zip(offsets, ys) if y == 1
        for i in range(max(a, start), min(b, start + len(needle)))
    }
    assert marked == set(range(start, start + len(needle))) and sample[start:start + len(needle)] == needle
    for _, _, offsets, ys in e["chunks"]:
        assert all(y != 1 or (a < start + len(needle) and start < b) for (a, b), y in zip(offsets, ys))

aug_probe = dict(
    id="aug-check",
    text='Chào bạn nhé!\nKhông được nói "đồ ngu" với người ta như vậy!',
    nhan="XUC_PHAM",
    bang_chung=["đồ ngu"],
    so_dong=2,
)
aug_rng = random.Random(SEED)
for _ in range(12):
    for aug_row in (augment_accents(aug_probe, aug_rng), augment_surface(aug_probe, aug_rng)):
        assert aug_row["bang_chung"] and all(any(p in l for l in aug_row["text"].splitlines()) for p in aug_row["bang_chung"])
        assert not re.search(r"[\u200b\u200c\u200d\ufeffаеосрухкѕіАЕОСРУХКВНМ]", aug_row["text"])
        prep_aug = prepare(aug_row)
        assert any(y == 1 for _, _, _, ys in prep_aug["chunks"] for y in ys)

encoded = {k: [prepare(r) for r in v] for k, v in parts.items()}
def batches(data, shuffle=False, epoch=0, batch_size=BATCH):
    if shuffle:
        return DataLoader(
            data,
            batch_sampler=BucketBatchSampler(data, batch_size, seed=SEED, epoch=epoch),
            collate_fn=collate,
            num_workers=0,
        )
    return DataLoader(data, batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=0)
def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
print("Số đoạn và số khối", {k: (len(v), sum(len(e["chunks"]) for e in v)) for k, v in encoded.items()})

---
<a id="sec-4"></a>
## 4. Kiến trúc Backbone trộn trọng số & Đầu phân cấp `GuardMerge`

Backbone sử dụng các lớp `Qwen3Model` tương thích cùng tên và hình dạng tensor với `BidirLM-0.6B-Base`, truyền mặt nạ **attention hai chiều tường minh (`full_attention`)**. Quá trình trộn trọng số với `Qwen3Guard-Gen-0.6B` chạy trong FP32 theo từng khối nhỏ trước khi chuyển BF16/FP16; riêng tầng embedding chỉ trộn các hàng có chung token ở hai tokenizer.

```text
  [BidirLM-0.6B-Base] + α·[Qwen3Guard-Gen-0.6B]  ──▶  Qwen3 Bidirectional Backbone (+ LoRA Attn/MLP)
                                                                       │
                                                                       ▼ (FP32 autocast disabled)
            ┌─────────────────────────────── GuardHierarchy (HEAD_ARCH="v4") ───────────────────────────────┐
            │  in_norm → project → [mean_pool + token_attn] → pos_emb → turn_attention → turn_ffn → Conv1d  │
            │       ├──▶ Nhánh Bằng chứng: Span Logits mức token (50% BCE + 50% Soft Dice)                  │
            │       └──▶ Nhánh Phân loại: evidence_pool / (mass_sum + 1) → Truy vấn 5 nhãn tình huống       │
            └───────────────────────────────────────────────────────────────────────────────────────────────┘
```

| Biến thể `HEAD_ARCH` | Mô tả cấu trúc đầu phân cấp (FP32) |
| :--- | :--- |
| **`"v4"`** *(đầy đủ, mặc định)* | `in_norm` → `project` → kết hợp `mean_pool` và `token_attn` (dùng phép max tương thích ONNX opset chuẩn, không dùng `scatter_reduce`) → `position` (`std=0.02`) → `turn_attention` → `turn_ffn` → `Conv1d(k=3)` có mặt nạ → nhánh dư `evidence_pool` chia theo `(mass_sum + 1)` → truy vấn theo nhãn. |
| **`"v3"`** *(tối giản cho ablation)* | Đầu phân cấp tối giản không có `turn_ffn`/`Conv1d` (`mean_pool`, khởi tạo mặc định, chuẩn hóa tín hiệu bằng chứng theo `counts.clamp_min(1)`). |

Cuối ô 4.1 tự động kiểm định **tính hai chiều của attention** và **tính bất biến tuyệt đối với padding** trên cả logits phân loại lẫn điểm bằng chứng mức token.

In [ ]:
#@title 4.1. Trộn Backbone, Khởi tạo GuardHierarchy & Kiểm định Padding/Hai chiều
# ==============================================================================
# 1. TẢI & TRỘN TRỌNG SỐ BIDIRLM + QWEN3GUARD
# ==============================================================================
def load_merged():
    bc = json.loads(Path(hub(BASE, "config.json", BREV)).read_text())
    gc = json.loads(Path(hub(GUARD, "config.json", GREV)).read_text()) if MERGE_RATIO else bc
    for k in (
        "vocab_size", "hidden_size", "intermediate_size", "num_hidden_layers", "head_dim",
        "num_attention_heads", "num_key_value_heads", "rope_theta", "rms_norm_eps",
    ):
        assert bc[k] == gc[k], k
    cfg = Qwen3Config(**{k: v for k, v in bc.items() if k not in {"architectures", "model_type", "auto_map", "dtype"}})
    cfg.use_cache = False
    cfg._attn_implementation = "sdpa"
    # Khởi tạo trực tiếp dtype đích để tránh bản sao FP32 toàn backbone.
    old = torch.get_default_dtype()
    try:
        torch.set_default_dtype(DTYPE)
        model = Qwen3Model(cfg)
    finally:
        torch.set_default_dtype(old)
    base_path = hub(BASE, "model.safetensors", BREV)
    guard_path = hub(GUARD, "model.safetensors", GREV) if MERGE_RATIO else None
    if guard_path:
        other = Tokenizer.from_file(hub(GUARD, "tokenizer.json", GREV)).get_vocab()
        shared = torch.zeros(cfg.vocab_size, dtype=torch.bool)
        for token, i in tok.get_vocab().items():
            if i < cfg.vocab_size and other.get(token) == i:
                shared[i] = True
    with safe_open(base_path, framework="pt") as base, (safe_open(guard_path, framework="pt") if guard_path else nullcontext()) as guard:
        expected = {"model." + name for name, _ in model.named_parameters()}
        assert expected == set(base.keys()), "Các tensor backbone khác revision đã kiểm tra"
        if guard:
            assert expected.issubset(guard.keys())
        with torch.no_grad():
            for name, p in model.named_parameters():
                key = "model." + name
                a = base.get_tensor(key)
                b = guard.get_tensor(key) if guard else None
                assert p.shape == a.shape and (b is None or b.shape == a.shape), key
                for s in range(0, len(a), 1024):
                    ix = slice(s, s + 1024)
                    value = a[ix].float()
                    if b is not None:
                        mixed = torch.lerp(value, b[ix].float(), MERGE_RATIO)
                        value = torch.where(shared[ix, None], mixed, value) if name == "embed_tokens.weight" else mixed
                    p[ix].copy_(value)
    return model


# ==============================================================================
# 2. BỘ ĐIỀU HỢP LORA (ATTENTION & SWIGLU MLP)
# ==============================================================================
class LoRALinear(nn.Module):
    def __init__(self, base, rank):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.in_features, rank, bias=False)
        self.up = nn.Linear(rank, base.out_features, bias=False)
        nn.init.zeros_(self.up.weight)
        self.scale = 2.0
        self.dropout = nn.Dropout(0.05)
    def forward(self, x):
        delta = self.up(self.down(self.dropout(x).to(self.down.weight.dtype)))
        return self.base(x) + self.scale * delta.to(x.dtype)


# ==============================================================================
# 3. ĐẦU PHÂN CẤP HỘI THOẠI GUARDHIERARCHY (FP32)
# ==============================================================================
class GuardHierarchy(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone.requires_grad_(False)
        for layer in self.backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name), RANK))
            if LORA_MLP:
                for name in ("gate_proj", "up_proj", "down_proj"):
                    setattr(layer.mlp, name, LoRALinear(getattr(layer.mlp, name), RANK))
        d_model = backbone.config.hidden_size
        if HEAD_ARCH == "v4":
            self.in_norm = nn.LayerNorm(d_model)
        self.project = nn.Linear(d_model, WIDTH)
        self.position = nn.Embedding(8, WIDTH)
        if HEAD_ARCH == "v4":
            nn.init.normal_(self.position.weight, std=0.02)
            self.token_attn = nn.Linear(WIDTH, 1)
        self.turn_attention = nn.MultiheadAttention(WIDTH, 4, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(WIDTH)
        if HEAD_ARCH == "v4":
            self.turn_ffn = nn.Sequential(
                nn.Linear(WIDTH, WIDTH * 2),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(WIDTH * 2, WIDTH),
            )
            self.norm_ffn = nn.LayerNorm(WIDTH)
            self.span_conv = nn.Conv1d(WIDTH, WIDTH, kernel_size=3, padding=1, groups=WIDTH, bias=False)
            self.span_mix = nn.Linear(WIDTH, WIDTH)
            self.span_norm = nn.LayerNorm(WIDTH)
            self.cue_norm = nn.LayerNorm(WIDTH)
        self.queries = nn.Parameter(torch.randn(len(LABELS), WIDTH) / math.sqrt(WIDTH))
        self.class_vectors = nn.Parameter(torch.randn_like(self.queries) / math.sqrt(WIDTH))
        self.bias = nn.Parameter(torch.zeros(len(LABELS)))
        if PROTOTYPE_WEIGHT:
            self.prototypes = nn.Parameter(torch.randn(len(LABELS), WIDTH) / math.sqrt(WIDTH))
        self.evidence = nn.Linear(WIDTH, 1)
        self.drop = nn.Dropout(0.1)

    def token_states(self, ids, mask):
        # Dạng dict bỏ qua bộ tạo causal mask của Qwen3. SDPA bool True = được chú ý.
        attn = mask[:, None, None, :].expand(-1, 1, ids.size(1), -1)
        return self.backbone(
            input_ids=ids,
            attention_mask={"full_attention": attn},
            use_cache=False,
        ).last_hidden_state

    def forward(self, b, with_aux=False):
        hidden = self.token_states(b["ids"], b["mask"])
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            x = self.in_norm(hidden.float()) if HEAD_ARCH == "v4" else hidden.float()
            h = self.project(x)
            valid = b["mask"].clone()
            valid[:, 0] = False
            n = len(b["labels"])
            idx = 8 * b["owner"] + b["turn"]
            h_valid = h * valid.unsqueeze(-1)
            sums = h.new_zeros(n * 8, WIDTH).index_add(0, idx, h_valid.sum(1))
            counts = h.new_zeros(n * 8).index_add(0, idx, valid.sum(1).to(h.dtype))
            present = counts.view(n, 8) > 0
            mean_pool = sums / counts.clamp_min(1)[:, None]

            if HEAD_ARCH == "v4":
                tok_score = self.token_attn(h).squeeze(-1).clamp(-20.0, 20.0).masked_fill(~valid, -1e4)
                slot_ids = torch.arange(n * 8, device=idx.device)
                belongs = idx[:, None] == slot_ids[None, :]
                chunk_max = tok_score.max(dim=1).values[:, None]
                slot_max = chunk_max.expand(-1, n * 8).masked_fill(~belongs, -1e4).max(dim=0).values
                w = torch.exp(tok_score - slot_max[idx][:, None]) * valid
                w_sum = h.new_zeros(n * 8).index_add(0, idx, w.sum(1))
                attn_sums = h.new_zeros(n * 8, WIDTH).index_add(0, idx, (w.unsqueeze(-1) * h).sum(1))
                attn_pool = attn_sums / w_sum.clamp_min(1e-6)[:, None]
                turns = (0.5 * mean_pool + 0.5 * attn_pool).view(n, 8, WIDTH)
            else:
                turns = mean_pool.view(n, 8, WIDTH)

            turns = turns + self.position.weight.to(h.dtype)
            context = self.turn_attention(turns, turns, turns, key_padding_mask=~present, need_weights=False)[0]
            turns = self.norm(turns + context)
            if HEAD_ARCH == "v4":
                turns = self.norm_ffn(turns + self.turn_ffn(turns))

            token_context = torch.tanh(h + turns[b["owner"], b["turn"]][:, None])
            if HEAD_ARCH == "v4":
                masked_ctx = token_context * valid.unsqueeze(-1)
                local = self.span_conv(masked_ctx.transpose(1, 2)).transpose(1, 2) * valid.unsqueeze(-1)
                span_feat = self.span_norm(token_context + F.gelu(self.span_mix(self.drop(local))))
            else:
                span_feat = token_context
            spans = self.evidence(self.drop(span_feat)).squeeze(-1).float()

            if EVIDENCE_POOL and SPAN_WEIGHT:
                mass = spans.sigmoid().to(h.dtype) * valid
                if HEAD_ARCH == "v4":
                    cue = h.new_zeros(n * 8, WIDTH).index_add(0, idx, (mass.unsqueeze(-1) * span_feat).sum(1))
                    mass_sum = h.new_zeros(n * 8).index_add(0, idx, mass.sum(1))
                    cue_vec = (cue / (mass_sum + 1.0)[:, None]).view(n, 8, WIDTH)
                    turns = self.cue_norm(turns + self.drop(cue_vec))
                else:
                    cue = h.new_zeros(n * 8, WIDTH).index_add(0, idx, (mass.unsqueeze(-1) * token_context).sum(1))
                    turns = turns + (cue / counts.clamp_min(1)[:, None]).view(n, 8, WIDTH)

            scores = torch.einsum("btd,ld->blt", turns, self.queries) / math.sqrt(WIDTH)
            att = scores.masked_fill(~present[:, None, :], float("-inf")).softmax(-1)
            pooled = torch.einsum("blt,btd->bld", att, turns)
            logits = (self.drop(pooled) * self.class_vectors).sum(-1) + self.bias
            if with_aux:
                proto = None
                if PROTOTYPE_WEIGHT:
                    doc = (turns * present.unsqueeze(-1)).sum(1) / present.sum(1, keepdim=True)
                    proto = F.normalize(doc.float(), dim=-1) @ F.normalize(self.prototypes.float(), dim=-1).T / 0.15
                return logits.float(), spans, proto
            return logits.float(), spans


# ==============================================================================
# 4. KHỞI TẠO MÔ HÌNH, NGỮ CẢNH AMP & KIỂM ĐỊNH HAI CHIỀU / PADDING
# ==============================================================================
def build_model():
    seed_all(SEED)
    model = GuardHierarchy(load_merged()).to(DEVICE)
    if GRAD_CKPT:
        model.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return model

def amp():
    return torch.autocast("cuda", dtype=DTYPE) if DEVICE.type == "cuda" else nullcontext()

model = build_model()
with torch.no_grad(), amp():
    model.eval()
    if HEAD_ARCH == "v4":
        assert 0.005 < float(model.position.weight.detach().float().std()) < 0.05
    b = to_device(next(iter(batches(encoded["fit"]))), DEVICE)
    logits, span_logits = model(b)
    assert logits.shape == (len(b["labels"]), 5) and torch.isfinite(logits).all()
    assert span_logits.shape == b["ids"].shape
    probe = torch.tensor([[BOS, 100, 101], [BOS, 100, 201]], device=DEVICE)
    state = model.token_states(probe, torch.ones_like(probe, dtype=torch.bool))
    assert (state[0, 1] - state[1, 1]).abs().max() > 1e-5, "Attention chưa thấy token tương lai"
    # Mẫu 1 lượt và nhiều lượt giữ nguyên dự đoán khi đứng cạnh mẫu dài có padding.
    short = prepare(dict(id="short", text="Cảm ơn bạn", nhan="AN_TOAN", bang_chung=[]))
    multi = prepare(dict(id="multi", text="Chào bạn nhé\nChiều gặp ở thư viện", nhan="AN_TOAN", bang_chung=[]))
    long = prepare(dict(id="long", text="Chiều nay mình gặp nhau ở thư viện trường lúc mấy giờ để làm tiếp bài tập nhóm vậy bạn?", nhan="AN_TOAN", bang_chung=[]))
    atol = {torch.float32: 1e-4, torch.float16: 1e-2, torch.bfloat16: 5e-2}[DTYPE]
    for target in (short, multi):
        single = to_device(collate([target]), DEVICE)
        padded = to_device(collate([target, long]), DEVICE)
        assert padded["ids"].size(1) > single["ids"].size(1)
        left, ls = model(single)
        right, rs = model(padded)
        nc = target["n_chunks"]
        torch.testing.assert_close(left[0], right[0], atol=atol, rtol=atol)
        torch.testing.assert_close(ls[:nc], rs[:nc, :ls.size(1)], atol=atol, rtol=atol)
    del b, logits, span_logits, probe, state, short, multi, long, target, single, padded, left, ls, right, rs
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("Số tham số huấn luyện", sum(p.numel() for p in model.parameters() if p.requires_grad))

---
<a id="sec-5"></a>
## 5. Huấn luyện mô hình, Dual-View JS Consistency & Chọn Checkpoint

### Bảng thành phần hàm mất mát & Điều chuẩn

| Thành phần | Cấu hình & Vai trò |
| :--- | :--- |
| **Trọng số phân tầng (`STRATUM_WEIGHT`)** | Xuất phát từ `(N / count[label]) ** 0.5` trên `fit`, tăng có kiểm soát cho nhóm câu thật hiếm (`QUAY_ROI|that`, `DE_DOA|that`), chặn tối đa ở `REAL_RARE_CAP = 2,5`. |
| **Dual-View JS Consistency (`CONSISTENCY_WEIGHT = 0,25`)** | Ghép cặp `(bản sạch, bản bỏ dấu/C1)` trong cùng microbatch để tối ưu phân kỳ JS trên xác suất 5 nhãn nhằm tăng độ bền trước biến dạng bề mặt và điều chuẩn chống quá khớp trên mẫu sinh. |
| **Hàm mất mát bằng chứng & Chuẩn hóa `#6`** | Trộn `50% BCE + 50% Soft Dice`, đo song song F1 mức từ và **F1 theo ý** ([`DO_DO_BANG_CHUNG.md`](../docs/v2/DO_DO_BANG_CHUNG.md)); tích hợp sẵn `infer_normalized` (chuẩn hóa 5 tầng giữ dòng `\n`, bảo đảm `non_verbatim == 0`). |

### Tiêu chí chọn Checkpoint & Quản lý bộ nhớ GPU
- **Chế độ `dev_80_10_10` (`CKPT_SELECT_MODE = "real_aligned"`):** Kết hợp `0.50 * macro_f1 + 0.15 * that_macro_f1 + 0.25 * that_acc + 0.10 * ev_f1` trên `dev` (hoặc `0.55 * macro_f1 + 0.20 * that_macro_f1 + 0.25 * that_acc` khi `SPAN_WEIGHT == 0`) nhằm cân bằng hiệu năng toàn cục, độ chính xác trên mẫu thật và chất lượng bằng chứng; `ckpt_select_weights` được ghi vào `config.json` để mỗi thư mục `RUN` tương ứng duy nhất với một cấu hình thực nghiệm.
- **Chế độ `final_90_10` / `final_100_human_cal`:** Huấn luyện đúng `EPOCHS` cố định, hoàn toàn không chạm `cal`.
- **Sạch VRAM trước benchmark:** Cuối ô giải phóng toàn bộ `opt`, `scheduler`, `scaler` và biến tạm khỏi GPU.

In [ ]:
#@title 5.1. Hàm mất mát, Suy luận/Chuẩn hóa #6, Độ đo bằng chứng & Huấn luyện
# ==============================================================================
# 1. TRỌNG SỐ PHÂN TẦNG & HÀM MẤT MÁT ĐA NHIỆM (CE / DICE / DUAL-VIEW JS)
# ==============================================================================
def compute_weights(data):
    lc = Counter(e["label"] for e in data)
    n_tot = len(data)
    raw_lbl = {k: (n_tot / lc[k]) ** 0.5 for k in range(len(LABELS))}
    mean_lbl = sum(raw_lbl[e["label"]] for e in data) / max(n_tot, 1)
    w_lbl = {k: v / mean_lbl for k, v in raw_lbl.items()}
    if STRATUM_WEIGHT:
        sc = Counter((e["label"], e["row"]["meta"]["loai"]) for e in data)
        raw = {}
        for (lbl, loai), c in sc.items():
            base = w_lbl[lbl]
            if loai == "that" and c < 0.25 * lc[lbl]:
                mult = min(2.0, (lc[lbl] / c) ** 0.25)
                raw[(lbl, loai)] = base * mult
            else:
                raw[(lbl, loai)] = base
        mw = sum(raw[(e["label"], e["row"]["meta"]["loai"])] for e in data) / max(n_tot, 1)
        sw = {k: v / mw for k, v in raw.items()}
        for _ in range(5):
            sw = {k: min(REAL_RARE_CAP, v) for k, v in sw.items()}
            mw = sum(sw[(e["label"], e["row"]["meta"]["loai"])] for e in data) / max(n_tot, 1)
            sw = {k: v / mw for k, v in sw.items()}
        sw = {k: min(REAL_RARE_CAP, v) for k, v in sw.items()}
    else:
        sw = w_lbl
    tags = Counter(y for e in data for _, _, _, ys in e["chunks"] for y in ys)
    ratio = tags[0] / max(tags[1], 1)
    if HEAD_ARCH == "v3" and SPAN_DICE_WEIGHT == 0.0:
        pw = torch.tensor(min(8.0, ratio), device=DEVICE)
    else:
        pw = torch.tensor(min(POS_WEIGHT_CAP, ratio ** 0.5), device=DEVICE)
    return sw, pw

stratum_weights, pos_weight = compute_weights(encoded["fit"])
print("Trọng số mẫu:", {str(k): round(v, 3) for k, v in sorted(stratum_weights.items())}, "| pos_weight:", round(float(pos_weight), 3))

def sample_weights(b):
    if STRATUM_WEIGHT:
        vals = [stratum_weights.get((e["label"], e["row"]["meta"]["loai"]), 1.0) for e in b["examples"]]
    else:
        vals = [stratum_weights.get(e["label"], 1.0) for e in b["examples"]]
    return torch.tensor(vals, dtype=torch.float32, device=DEVICE)

def accumulation_windows(loader, size):
    iterator = iter(loader)
    while window := list(islice(iterator, size)):
        yield window

def loss_denominators(sub_batches):
    class_mass = sum(sample_weights(b).sum() for b in sub_batches)
    known_count = sum(int((b["targets"] != -100).sum()) for b in sub_batches)
    pair_count = sum(len(b.get("pairs") or ()) for b in sub_batches)
    return class_mass.clamp_min(1e-6), max(known_count, 1), max(pair_count, 1)

def js_divergence_probs(p, q, eps=1e-6):
    p = p.clamp(eps, 1.0 - eps)
    q = q.clamp(eps, 1.0 - eps)
    m = 0.5 * (p + q)
    kl_pm = (p * (p.log() - m.log())).sum(dim=-1)
    kl_qm = (q * (q.log() - m.log())).sum(dim=-1)
    return 0.5 * (kl_pm + kl_qm)

def objective(logits, spans, proto, b, class_mass, known_count, pair_count=None):
    sw = sample_weights(b)
    def class_loss(values):
        ce = F.cross_entropy(values.float(), b["labels"], reduction="none", label_smoothing=LABEL_SMOOTHING)
        return (ce * sw).sum() / class_mass
    loss = class_loss(logits)
    if SPAN_WEIGHT:
        known = b["targets"] != -100
        targets = b["targets"].clamp_min(0).float()
        bce = F.binary_cross_entropy_with_logits(
            spans.float(), targets, pos_weight=pos_weight, reduction="none"
        ).masked_fill(~known, 0).sum() / known_count
        if SPAN_DICE_WEIGHT and known.any():
            probs = spans.float().sigmoid()
            km = known.float()
            inter = (probs * targets * km).sum()
            denom = (probs * km).sum() + (targets * km).sum()
            dice = (1.0 - (2.0 * inter + 2.0) / (denom + 2.0)) * (km.sum() / known_count)
            span_loss = (1.0 - SPAN_DICE_WEIGHT) * bce + SPAN_DICE_WEIGHT * dice
        else:
            span_loss = bce
        loss = loss + SPAN_WEIGHT * span_loss
    if proto is not None:
        loss = loss + PROTOTYPE_WEIGHT * class_loss(proto)
    if CONSISTENCY_WEIGHT > 0 and b.get("pairs"):
        i0 = torch.tensor([p[0] for p in b["pairs"]], device=logits.device)
        i1 = torch.tensor([p[1] for p in b["pairs"]], device=logits.device)
        p0 = F.softmax(logits[i0].float(), dim=-1)
        p1 = F.softmax(logits[i1].float(), dim=-1)
        pair_denom = max(pair_count if pair_count is not None else len(b["pairs"]), 1)
        loss = loss + CONSISTENCY_WEIGHT * js_divergence_probs(p0, p1).sum() / pair_denom
    return loss


# ==============================================================================
# 2. SUY LUẬN GỐC (#5) & BỘ CHUẨN HÓA 5 TẦNG ÁNH XẠ NGƯỢC VERBATIM (#6)
# ==============================================================================
def word_record(row, pieces, logits):
    ys, unknown = annotations(row)
    words = []
    first = 0
    for m in re.finditer(r"\S+", row["text"]):
        loc = m.span()
        while first < len(pieces) and pieces[first][1] <= loc[0]:
            first += 1
        vals = []
        j = first
        while j < len(pieces) and pieces[j][0] < loc[1]:
            a, b, value = pieces[j]
            if a < b and b > loc[0]:
                vals.append(value)
            j += 1
        score = (max(vals) if WORD_SCORE == "max" else sum(vals) / len(vals)) if vals else 0.0
        target = int(overlaps(loc, ys))
        if overlaps(loc, unknown) or (row["nhan"] != "AN_TOAN" and not row["bang_chung"]):
            target = -1
        words.append((loc[0], loc[1], score, target))
    return dict(
        id=row["id"],
        text=row["text"],
        gold=LABELS.index(row["nhan"]),
        gold_phrases=list(row.get("bang_chung") or []),
        logits=logits,
        words=words,
    )

@torch.inference_mode()
def infer(net, data, device=DEVICE, batch_size=EVAL_BATCH):
    net.eval()
    out = []
    for batch in batches(data, batch_size=batch_size):
        for sub in split_microbatch(batch):
            b = to_device(sub, device)
            with (amp() if device.type == "cuda" else nullcontext()):
                logits, spans = net(b)
            scores = spans.sigmoid().cpu().tolist()
            by_doc = [[] for _ in sub["examples"]]
            for j, (i, off) in enumerate(zip(sub["owner"].tolist(), sub["offsets"])):
                by_doc[i].extend((a, z, s) for (a, z), s in zip(off, scores[j][1:]))
            out.extend(word_record(e["row"], p, l) for e, p, l in zip(sub["examples"], by_doc, logits.cpu().tolist()))
    return out

_NORM_LOOKALIKE = {"4": "a", "3": "e", "1": "i", "0": "o", "5": "s", "9": "g", "8": "b", "@": "a"}
_NORM_SEP_RE = re.compile(r"(?<=[a-zA-ZÀ-ỹ])[.\-_·~](?=[a-zA-ZÀ-ỹ])")
_NORM_ELONG_RE = re.compile(r"([a-zA-ZÀ-ỹ])\1{2,}")
_NORM_LEAD_RE = re.compile(r"^([^\w]*)")
_NORM_TRAIL_RE = re.compile(r"([^\w]*)$")

def _norm_strip_base(ch):
    s = ch.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s) if not unicodedata.combining(c))

def _normalize_v1_token(tok_str, teencode_map):
    t = unicodedata.normalize("NFC", tok_str)
    if t and not t[0].isdigit():
        na = sum(1 for c in t if c.isalpha())
        nc = sum(1 for c in t if c in _NORM_LOOKALIKE)
        if nc > 0 and na > nc:
            t = "".join(_NORM_LOOKALIKE.get(c, c) for c in t)
    t = _NORM_SEP_RE.sub("", t)
    t = _NORM_ELONG_RE.sub(r"\1", t)
    if len(t) >= 2:
        chars = list(t)
        res = [chars[0]]
        for i in range(1, len(chars)):
            p, c = chars[i - 1], chars[i]
            if p != c and c.isalpha() and p.isalpha() and _norm_strip_base(p) == _norm_strip_base(c) and p != _norm_strip_base(p):
                continue
            res.append(c)
        t = "".join(res)
    lead = _NORM_LEAD_RE.match(t).group(1)
    trail = _NORM_TRAIL_RE.search(t).group(1)
    core = t[len(lead):len(t) - len(trail)] if trail else t[len(lead):]
    if core and teencode_map:
        rep = teencode_map.get(core.lower())
        if rep:
            if core[0].isupper() and rep[0].islower():
                rep = rep[0].upper() + rep[1:]
            t = lead + rep + trail
    return t.strip() or tok_str

def normalize_with_alignment(text, teencode_map):
    """Chuẩn hóa 5 tầng kiểu v1 theo từng dòng (giữ nguyên \n) và ánh xạ ngược mỗi từ về chỉ số từ văn bản gốc."""
    norm_lines, norm_to_orig, orig_idx = [], [], 0
    for line in text.split("\n"):
        out_words = []
        for m in re.finditer(r"\S+", line):
            nt = _normalize_v1_token(m.group(0), teencode_map)
            pieces = nt.split() or [m.group(0)]
            for p in pieces:
                out_words.append(p)
                norm_to_orig.append(orig_idx)
            orig_idx += 1
        norm_lines.append(" ".join(out_words))
    return "\n".join(norm_lines), norm_to_orig

def infer_normalized(net, records, teencode_map, infer_fn=None, prep_fn=None):
    """Suy luận Dòng #6 (#5 + chuẩn hóa đầu vào giữ dòng) và chiếu ngược điểm bằng chứng về văn bản gốc (100% verbatim)."""
    _prep = prepare if prep_fn is None else prep_fn
    _inf = infer if infer_fn is None else infer_fn
    norm_encoded, mappings = [], []
    for r in records:
        norm_text, norm_to_orig = normalize_with_alignment(r["text"], teencode_map)
        norm_encoded.append(_prep(dict(r, text=norm_text, bang_chung=[])))
        mappings.append(norm_to_orig)
    raw_preds = _inf(net, norm_encoded)
    projected = []
    for r, p, norm_to_orig in zip(records, raw_preds, mappings):
        ys, unknown = annotations(r)
        skip = r["nhan"] != "AN_TOAN" and not r["bang_chung"]
        orig_spans = []
        for m in re.finditer(r"\S+", r["text"]):
            loc = m.span()
            target = -1 if (skip or overlaps(loc, unknown)) else int(overlaps(loc, ys))
            orig_spans.append((loc[0], loc[1], target))
        by_orig = defaultdict(list)
        n_pw = len(p["words"])
        for j, oi in enumerate(norm_to_orig):
            if j < n_pw:
                by_orig[oi].append(j)
        orig_words = []
        for oi, (a, b, target) in enumerate(orig_spans):
            js = by_orig.get(oi)
            if js:
                vals = [p["words"][j][2] for j in js]
                sc = float(max(vals) if WORD_SCORE == "max" else sum(vals) / len(vals))
            else:
                sc = 0.0
            orig_words.append((a, b, sc, int(target)))
        projected.append(dict(
            id=r["id"],
            text=r["text"],
            gold=LABELS.index(r["nhan"]),
            gold_phrases=list(r.get("bang_chung") or []),
            logits=p["logits"],
            words=orig_words,
        ))
    return projected


# ==============================================================================
# 3. HẬU XỬ LÝ BẰNG CHỨNG, ĐỘ ĐO F1 THEO Ý & TIÊU CHÍ CHỌN CHECKPOINT
# ==============================================================================
def _apply_empty_fallback(marks, words, text, fallback_min_prob=None, expand_ratio=0.70, max_words=5):
    """Fallback có ngưỡng tin cậy: chọn cụm đỉnh cùng dòng khi mẫu đã dự đoán nguy hại nhưng marks rỗng
    và từ có xác suất cao nhất >= fallback_min_prob."""
    if any(marks) or not words or fallback_min_prob is None or fallback_min_prob > 1.0:
        return marks
    probs = [w[2] for w in words]
    peak_i = int(np.argmax(probs))
    peak_p = probs[peak_i]
    if peak_p < fallback_min_prob:
        return marks
    out = list(marks)
    out[peak_i] = True
    cut = peak_p * expand_ratio
    lo = hi = peak_i
    while (hi - lo + 1) < max_words:
        can_l = lo > 0 and "\n" not in text[words[lo - 1][1]:words[lo][0]] and probs[lo - 1] >= cut
        can_r = hi + 1 < len(words) and "\n" not in text[words[hi][1]:words[hi + 1][0]] and probs[hi + 1] >= cut
        if not can_l and not can_r:
            break
        if can_l and (not can_r or probs[lo - 1] >= probs[hi + 1]):
            lo -= 1
            out[lo] = True
        else:
            hi += 1
            out[hi] = True
    return out

def select_word_marks(item, threshold=0.5, single_word_threshold=None, bridge_ratio=0.0, gate=True, fallback_min_prob=None):
    words = item["words"]
    n_w = len(words)
    if not n_w or (gate and int(np.argmax(item["logits"])) == 0):
        return [False] * n_w
    marks = [score >= threshold for _, _, score, _ in words]
    text = item["text"]
    if bridge_ratio > 0 and n_w >= 3:
        bridge_thr = threshold * bridge_ratio
        bridged = marks[:]
        for i in range(1, n_w - 1):
            if not marks[i] and marks[i - 1] and marks[i + 1] and words[i][2] >= bridge_thr:
                if "\n" not in text[words[i - 1][1]:words[i + 1][0]]:
                    bridged[i] = True
        marks = bridged
    if single_word_threshold is not None and single_word_threshold > threshold:
        filtered = marks[:]
        for i in range(n_w):
            if not marks[i]:
                continue
            left_same = i > 0 and marks[i - 1] and "\n" not in text[words[i - 1][1]:words[i][0]]
            right_same = i + 1 < n_w and marks[i + 1] and "\n" not in text[words[i][1]:words[i + 1][0]]
            if not left_same and not right_same and words[i][2] < single_word_threshold:
                filtered[i] = False
        marks = filtered
    if gate and fallback_min_prob is not None:
        marks = _apply_empty_fallback(marks, words, text, fallback_min_prob=fallback_min_prob)
    return marks

_CLAUSE_END_RE = re.compile(r"[,.;:!?…]+[\"'”’\)\]\}]*$")

def _word_line_clauses(text, words):
    out, line_id, clause_id, prev_end = [], 0, 0, 0
    for w in words:
        a, b = w[0], w[1]
        gap = text[prev_end:a]
        if "\n" in gap:
            line_id += gap.count("\n")
            clause_id += 1
        out.append((line_id, clause_id))
        if _CLAUSE_END_RE.search(text[a:b]):
            clause_id += 1
        prev_end = b
    return out

def f1_theo_y_item(text, words, marks, gold_phrases, margin=2):
    """Độ đo F1 theo ý (docs/v2/DO_DO_BANG_CHUNG.md): ±margin từ trong cùng mệnh đề và cùng dòng."""
    if not gold_phrases or not words:
        return None
    lc = _word_line_clauses(text, words)
    lines = text.split("\n")
    line_starts, pos = [], 0
    for ln in lines:
        line_starts.append(pos)
        pos += len(ln) + 1
    gold_groups, allowed = [], set()
    for ph in dict.fromkeys(p for p in gold_phrases if p):
        phrase_sets = []
        for l_idx, ln in enumerate(lines):
            base = line_starts[l_idx]
            for m in re.finditer("(?=" + re.escape(ph) + ")", ln):
                sa, sb = base + m.start(), base + m.start() + len(ph)
                g = [i for i, w in enumerate(words) if w[0] < sb and sa < w[1]]
                if not g:
                    continue
                phrase_sets.append(set(g))
                lo, hi = g[0], g[-1]
                allowed.update(g)
                lo_lc, hi_lc = lc[lo], lc[hi]
                idx, step = lo - 1, 0
                while idx >= 0 and step < margin and lc[idx] == lo_lc:
                    allowed.add(idx); idx -= 1; step += 1
                idx, step = hi + 1, 0
                while idx < len(words) and step < margin and lc[idx] == hi_lc:
                    allowed.add(idx); idx += 1; step += 1
        if phrase_sets:
            gold_groups.append(phrase_sets)
    if not gold_groups:
        return None
    pred_set = {i for i, m in enumerate(marks) if m}
    if not pred_set:
        return 0.0
    rec = sum(max(len(g & pred_set) / len(g) for g in g_list) for g_list in gold_groups) / len(gold_groups)
    prec = len(pred_set & allowed) / len(pred_set)
    return 2.0 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def evidence_stats(
    pred, threshold=0.5, gate=True, single_word_threshold=None, bridge_ratio=0.0,
    fallback_min_prob=None, sample_weights=None,
):
    tp = fp = fn = safe_mark = safe_count = 0.0
    harm_pred = harm_empty = 0.0
    y_f1_sum = y_f1_w = 0.0
    for idx_i, item in enumerate(pred):
        wi = 1.0 if sample_weights is None else float(sample_weights[idx_i])
        marks = select_word_marks(
            item,
            threshold=threshold,
            single_word_threshold=single_word_threshold,
            bridge_ratio=bridge_ratio,
            gate=gate,
            fallback_min_prob=fallback_min_prob,
        )
        if item["gold"] == 0:
            safe_count += wi
            safe_mark += wi * int(any(marks))
        else:
            yi = f1_theo_y_item(item["text"], item["words"], marks, item.get("gold_phrases") or [])
            if yi is not None:
                y_f1_sum += wi * yi
                y_f1_w += wi
        if int(np.argmax(item["logits"])) != 0:
            harm_pred += wi
            harm_empty += wi * int(not any(marks))
        for marked, (_, _, _, gold) in zip(marks, item["words"]):
            if gold < 0:
                continue
            tp += wi * int(marked and gold == 1)
            fp += wi * int(marked and gold == 0)
            fn += wi * int(not marked and gold == 1)
    return dict(
        precision=float(tp / max(tp + fp, 1e-12)),
        recall=float(tp / max(tp + fn, 1e-12)),
        f1=float(2 * tp / max(2 * tp + fp + fn, 1e-12)),
        f1_theo_y=float(y_f1_sum / max(y_f1_w, 1e-12)) if y_f1_w > 0 else 0.0,
        safe_mark_rate=float(safe_mark / max(safe_count, 1e-12)),
        word_length_ratio=float((tp + fp) / max(tp + fn, 1e-12)),
        harm_empty_evidence_rate=float(harm_empty / max(harm_pred, 1e-12)),
    )

def macro(pred):
    return f1_score(
        [p["gold"] for p in pred],
        [np.argmax(p["logits"]) for p in pred],
        labels=range(len(LABELS)),
        average="macro",
        zero_division=0,
    )

def slice_summary(pred, records):
    y = np.array([p["gold"] for p in pred])
    yp = np.array([int(np.argmax(p["logits"])) for p in pred])
    out = {}
    for src in ("that", "sinh"):
        ix = [i for i, r in enumerate(records) if (r.get("meta") or {}).get("loai") == src]
        if not ix:
            continue
        present = np.unique(y[ix])
        out[f"{src}_macro_f1"] = float(f1_score(y[ix], yp[ix], average="macro", labels=present, zero_division=0))
        if src == "that":
            out["that_acc"] = float((y[ix] == yp[ix]).mean())
            for lbl in ("QUAY_ROI", "DE_DOA"):
                j = LABELS.index(lbl)
                sub = [i for i in ix if y[i] == j]
                out[f"that_{lbl.lower()}_recall"] = float((yp[sub] == j).mean()) if sub else None
        else:
            ix_any = [i for i in ix if (records[i].get("meta") or {}).get("do_lo") == "an_y"]
            if ix_any:
                p_any = np.unique(y[ix_any])
                out["sinh_an_y_macro_f1"] = float(f1_score(y[ix_any], yp[ix_any], average="macro", labels=p_any, zero_division=0))
    return out

def checkpoint_select_key(macro_f1, slices_dict, ev_f1, span_weight=SPAN_WEIGHT, mode=CKPT_SELECT_MODE):
    if mode not in {"real_aligned", "macro_only"}:
        raise ValueError(f"CKPT_SELECT_MODE không hợp lệ: {mode!r}")
    m_val = float(macro_f1)
    ev_val = float(ev_f1) if span_weight > 0 else 0.0
    if mode == "macro_only":
        return (m_val, ev_val)
    sl = slices_dict or {}
    that_f1 = float(sl["that_macro_f1"]) if sl.get("that_macro_f1") is not None else m_val
    that_acc = float(sl["that_acc"]) if sl.get("that_acc") is not None else m_val
    if span_weight > 0:
        score = 0.50 * m_val + 0.15 * that_f1 + 0.25 * that_acc + 0.10 * ev_val
    else:
        score = 0.55 * m_val + 0.20 * that_f1 + 0.25 * that_acc
    return (round(score, 6), round(m_val, 6), round(that_acc, 6), round(that_f1, 6), round(ev_val, 6))


# ==============================================================================
# 4. LƯU / KHÔI PHỤC CHECKPOINT & DỌN SẠCH BỘ NHỚ GPU
# ==============================================================================
def trainable(net):
    return {k: p.detach().cpu().clone().contiguous() for k, p in net.named_parameters() if p.requires_grad}

def restore(net, state):
    params = {k: p for k, p in net.named_parameters() if p.requires_grad}
    assert set(params) == set(state)
    with torch.no_grad():
        for key, p in params.items():
            p.copy_(state[key])

def free_training_memory(net, ns):
    """Giải phóng hoàn toàn optimizer states, scaler, gradient và biến tạm khỏi GPU trước khi đo benchmark()."""
    net.zero_grad(set_to_none=True)
    for var_name in (
        "opt", "opt_groups", "scheduler", "scaler", "named",
        "sub_batches", "window", "b", "sub", "running", "window_loss",
        "loss", "fit_items", "loader",
    ):
        obj = ns.pop(var_name, None)
        if obj is not None and hasattr(obj, "zero_grad"):
            try:
                obj.zero_grad(set_to_none=True)
            except Exception:
                pass
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
        torch.cuda.empty_cache()

def is_no_decay(name, param):
    return param.ndim < 2 or name.endswith(".bias") or "norm" in name or "position" in name

named = [(k, p) for k, p in model.named_parameters() if p.requires_grad]
opt_groups = [
    {"params": [p for k, p in named if k.startswith("backbone.") and not is_no_decay(k, p)], "lr": LR_ADAPTER, "weight_decay": 0.01},
    {"params": [p for k, p in named if k.startswith("backbone.") and is_no_decay(k, p)], "lr": LR_ADAPTER, "weight_decay": 0.0},
    {"params": [p for k, p in named if not k.startswith("backbone.") and not is_no_decay(k, p)], "lr": LR_HEAD, "weight_decay": 0.01},
    {"params": [p for k, p in named if not k.startswith("backbone.") and is_no_decay(k, p)], "lr": LR_HEAD, "weight_decay": 0.0},
]
opt = torch.optim.AdamW([g for g in opt_groups if g["params"]])

accented_fit_idx = [i for i, r in enumerate(parts["fit"]) if strip_accents(r["text"]) != r["text"]]
accent_aug_count = min(len(accented_fit_idx), round(len(parts["fit"]) * ACCENT_AUG))
surface_aug_count = round(len(parts["fit"]) * SURFACE_AUG)


# ==============================================================================
# 5. VÒNG LẶP HUẤN LUYỆN, LƯU CHECKPOINT & GIẢI PHÓNG VRAM
# ==============================================================================
def build_epoch_items(epoch_rng):
    n_fit = len(parts["fit"])
    acc_pick = set(epoch_rng.sample(accented_fit_idx, accent_aug_count)) if accent_aug_count else set()
    rem_pool = [i for i in range(n_fit) if i not in acc_pick]
    if len(rem_pool) >= surface_aug_count:
        surf_pick = set(epoch_rng.sample(rem_pool, surface_aug_count)) if surface_aug_count else set()
    else:
        surf_pick = set(epoch_rng.sample(range(n_fit), surface_aug_count)) if surface_aug_count else set()
    items = []
    for i, e_clean in enumerate(encoded["fit"]):
        if i in acc_pick:
            e_pert = prepare(augment_accents(parts["fit"][i], epoch_rng))
            items.append((e_clean, e_pert))
            if i in surf_pick:
                items.append(prepare(augment_surface(parts["fit"][i], epoch_rng)))
        elif i in surf_pick:
            e_pert = prepare(augment_surface(parts["fit"][i], epoch_rng))
            items.append((e_clean, e_pert))
        else:
            items.append(e_clean)
    return items

probe_items = build_epoch_items(random.Random(SEED + 1000))
steps_epoch = math.ceil(math.ceil(len(probe_items) / BATCH) / ACCUM)
del probe_items
steps = steps_epoch * EPOCHS
warm = max(1, round(steps * 0.08))
def lr_factor(i):
    if i < warm:
        return (i + 1) / warm
    progress = min(1.0, (i - warm) / max(1, steps - warm))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda" and DTYPE == torch.float16)

last = RUN / "last.pt"
start_epoch, history, best_state, best_key = 0, [], {}, (-1.0,)
if last.exists():
    saved = torch.load(last, map_location="cpu", weights_only=True)
    assert saved["config"] == config
    start_epoch, history = saved["epoch"] + 1, saved["history"]
    best_state, best_key = saved["best"], tuple(saved["key"])
    if start_epoch < EPOCHS:
        restore(model, saved["trainable"])
        opt.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
    else:
        print(f"Đã huấn luyện đủ {EPOCHS} epoch trong {last}; khôi phục trực tiếp best_state (bỏ qua nạp optimizer states lên GPU).")
    del saved

has_dev = "dev" in encoded
dev_records = [e["row"] for e in encoded["dev"]] if has_dev else []
for epoch in range(start_epoch, EPOCHS):
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    epoch_start = time.perf_counter()
    seed_all(SEED + 100 + epoch)
    epoch_rng = random.Random(SEED + 1000 + epoch)
    fit_items = build_epoch_items(epoch_rng)
    loader = batches(fit_items, shuffle=True, epoch=epoch)
    model.train()
    opt.zero_grad(set_to_none=True)
    running = torch.zeros((), device=DEVICE)
    for step, window in enumerate(accumulation_windows(loader, ACCUM), start=1):
        sub_batches = [sb for batch in window for sb in split_microbatch(batch)]
        class_mass, known_count, pair_count = loss_denominators(sub_batches)
        window_loss = torch.zeros((), device=DEVICE)
        for sub in sub_batches:
            b = to_device(sub, DEVICE)
            with amp():
                loss = objective(*model(b, with_aux=True), b, class_mass, known_count, pair_count)
            scaler.scale(loss).backward()
            window_loss += loss.detach()
        if not torch.isfinite(window_loss):
            raise FloatingPointError(f"Loss lỗi ở epoch {epoch + 1}, bước {step}")
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for _, p in named], 1.0)
        old_scale = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        if scaler.get_scale() >= old_scale:
            scheduler.step()
        opt.zero_grad(set_to_none=True)
        running += window_loss
        if step % 32 == 0 or step == steps_epoch:
            print(f"epoch {epoch + 1}/{EPOCHS} | step {step}/{steps_epoch} | loss {running.item() / step:.4f}", flush=True)
    if has_dev:
        dev = infer(model, encoded["dev"])
        ev = evidence_stats(dev) if SPAN_WEIGHT else dict(precision=0.0, recall=0.0, f1=0.0, f1_theo_y=0.0, safe_mark_rate=0.0)
        m_f1 = float(macro(dev))
        sl = slice_summary(dev, dev_records)
        key = checkpoint_select_key(m_f1, sl, ev["f1"], span_weight=SPAN_WEIGHT, mode=CKPT_SELECT_MODE)
        if key > best_key:
            best_key, best_state = key, trainable(model)
        entry = dict(
            epoch=epoch + 1,
            train_loss=running.item() / steps_epoch,
            select_score=key[0],
            macro_f1=m_f1,
            **sl,
            evidence_f1=float(ev["f1"]),
            evidence_f1_theo_y=float(ev["f1_theo_y"]),
            evidence_precision=float(ev["precision"]),
            evidence_recall=float(ev["recall"]),
            safe_mark_rate=float(ev["safe_mark_rate"]),
            seconds=time.perf_counter() - epoch_start,
            peak_cuda_MiB=(torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == "cuda" else None),
        )
    else:
        best_key = (float(epoch + 1),)
        best_state = trainable(model)
        entry = dict(
            epoch=epoch + 1,
            train_loss=running.item() / steps_epoch,
            mode=f"fixed_epoch_{DATA_SCHEME}",
            seconds=time.perf_counter() - epoch_start,
            peak_cuda_MiB=(torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == "cuda" else None),
        )
    history.append(entry)
    tmp = last.with_suffix(".tmp")
    torch.save(
        dict(
            config=config,
            epoch=epoch,
            trainable=trainable(model),
            optimizer=opt.state_dict(),
            scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            history=history,
            best=best_state,
            key=best_key,
        ),
        tmp,
    )
    os.replace(tmp, last)
    print(entry, flush=True)

restore(model, best_state)
tmp = RUN / "adapter.tmp"
save_file(best_state, str(tmp))
os.replace(tmp, RUN / "adapter.safetensors")
write_json(RUN / "history.json", history)
free_training_memory(model, globals())
print(pd.DataFrame(history))
if DEVICE.type == "cuda":
    print(f"Sau khi giải phóng optimizer, VRAM đang giữ: {torch.cuda.memory_allocated(DEVICE) / 2**20:.1f} MiB")

---
<a id="sec-6"></a>
## 6. Hiệu chuẩn xác suất, Đánh giá phát triển & Cổng khóa Bộ Test (`C0 / C1 / C2`)

Ô này thực thi tuần tự quy trình hiệu chuẩn sát phân phối thật, báo cáo kết quả `dev` và cổng khóa cứng 1 lần duy nhất cho Bộ Test người viết (`1.670` mẫu):

| Giai đoạn | Quy tắc thực thi & Tệp xuất |
| :--- | :--- |
| **1. Hiệu chuẩn sát phân phối thật (`cal` / `HUMAN_CAL_PATH`)** | Nhiệt độ `T` dùng 70% câu thật + 30% câu sinh khó với trần ngoại lai `cap = 3.0` và biên `T >= 0.74` trên `internal_llm`; ngưỡng bằng chứng cân bằng 5 lớp. Lưu đóng băng vào `calibration.json`. |
| **2. Hậu xử lý bằng chứng chống phình cụm** | Dò bộ tham số `(span_threshold, single_word_threshold, bridge_ratio, fallback_min_prob)` trên `cal` có phạt phình độ dài (`word_length_ratio > 1.25`), báo song song F1 mức từ & **F1 theo ý** ([`DO_DO_BANG_CHUNG.md`](../docs/v2/DO_DO_BANG_CHUNG.md)). |
| **3. Báo cáo `dev` & Biểu đồ chẩn đoán** | Xuất `dev_metrics.json` (kèm Bootstrap 95% CI, `FPR(AN_TOAN)`, `worst_group_region_macro_f1`), `dev_predictions.jsonl` (đủ `probs` 5 lớp) và `dev_diagnostics.png`. |

> **CẢNH BÁO KHÓA TEST (`REAL_TEST_LOCK.json`):** Mặc định `EVALUATE_REAL_TEST = False`. Chỉ đặt `True` **đúng 1 lần duy nhất** sau khi hoàn tất lựa chọn cấu hình trên `dev` và đóng băng `calibration.json` (tự xác minh SHA-256, không ghi đè ngưỡng). Khi mở khóa, hệ thống chấm trọn `334 C0 + 668 C1 + 668 C2 = 1.670` mẫu cho cả **Dòng `#5` (thô)** lẫn **Dòng `#6` (+ chuẩn hóa giữ dòng)**, xuất `real_test_all_predictions.jsonl` / `real_test_norm_all_predictions.jsonl` (tương thích 100% `python -m src.v2.eval.evaluate`), ma trận 15 lát cắt `real_test_official_matrix.json`, và tự khóa `REAL_TEST_LOCK.json`.

In [ ]:
#@title 6.1. Hiệu chuẩn xác suất/ngưỡng (cal), Đánh giá dev & Chấm Test khóa 1 lần cuối
# ==============================================================================
# 1. CHỐNG RÒ RỈ DỮ LIỆU & TRỌNG SỐ HIỆU CHUẨN SÁT PHÂN PHỐI THẬT
# ==============================================================================
def _norm_dedup_text(s):
    return re.sub(r"[\W_]+", " ", strip_accents(s).lower()).strip()

def read_human(path):
    raw_bytes = Path(path).read_bytes()
    for locked_name in ("test.jsonl", "test_day_du.jsonl", "test_perturb_C1.jsonl", "test_perturb_C2.jsonl"):
        if _matches_blob_sha(raw_bytes, DATA_SHA256[locked_name]):
            raise ValueError(f"KHÓA AN TOÀN: HUMAN_CAL_PATH trùng SHA-256 của tập test thật ({locked_name}) — từ chối để chống rò rỉ!")
    for train_name in ("so_sanh.jsonl", "train.jsonl"):
        if _matches_blob_sha(raw_bytes, DATA_SHA256[train_name]):
            raise ValueError(f"KHÓA AN TOÀN: HUMAN_CAL_PATH trùng SHA-256 của tập train ({train_name}) — từ chối để chống rò rỉ!")
    gold = [json.loads(line) for line in raw_bytes.decode("utf-8-sig").splitlines() if line.strip()]
    training_texts = {r["text"].strip().casefold() for r in rows}
    training_norms = {nt for nt in (_norm_dedup_text(r["text"]) for r in rows) if nt}
    training_ids = {r["id"] for r in rows}
    assert gold and all(isinstance(r, dict) and r.get("id") for r in gold) and len({r["id"] for r in gold}) == len(gold), (
        "HUMAN_CAL_PATH rỗng, sai cấu trúc hoặc trùng id!"
    )
    seen_texts = set()
    for r in gold:
        assert isinstance(r.get("text"), str) and r["text"].strip(), f"Thiếu hoặc rỗng text tại id={r['id']}"
        cf = r["text"].strip().casefold()
        nt = _norm_dedup_text(r["text"])
        assert r["id"] not in training_ids and cf not in training_texts and (not nt or nt not in training_norms), (
            f"Rò rỉ dữ liệu train vào HUMAN_CAL_PATH tại id={r['id']}"
        )
        assert cf not in seen_texts, f"Trùng lặp nội dung bên trong HUMAN_CAL_PATH tại id={r['id']}"
        seen_texts.add(cf)
        bc = r.get("bang_chung")
        assert isinstance(bc, list) and all(isinstance(p, str) and p and "\n" not in p and any(p in ln for ln in r["text"].split("\n")) for p in bc), (
            f"Trường bang_chung không hợp lệ tại id={r['id']}"
        )
        assert r["nhan"] in LABELS
        assert r["nhan"] != "AN_TOAN" or not bc
        assert r.get("nguon") != "sinh_llm" and r.get("cach_gan") != "llm"
        r["so_dong"] = len(r["text"].split("\n"))
        assert 1 <= r["so_dong"] <= 8
        r.setdefault("nguon", "nguoi")
        if not r.get("nhom_trung"):
            r["nhom_trung"] = r["id"]
        meta = dict(r.get("meta") or {})
        meta.update({"loai": "that", "nguon": r["nguon"]})
        r["meta"] = meta
    return gold

def compute_cal_weights(records, mode=CAL_WEIGHT_MODE):
    if mode != "real_aligned" or not records:
        return np.ones(len(records), dtype=np.float64)
    raw = []
    for r in records:
        m = r.get("meta") or {}
        if m.get("loai", "that") == "that":
            raw.append(3.0)
        elif m.get("do_lo") == "an_y":
            raw.append(1.25)
        elif m.get("sinh_doi"):
            raw.append(0.85)
        else:
            raw.append(0.50)
    raw = np.asarray(raw, dtype=np.float64)
    labels = [r["nhan"] for r in records]
    present = sorted(set(labels))
    out = np.zeros_like(raw)
    for lbl in present:
        ix = np.array([i for i, y_l in enumerate(labels) if y_l == lbl], dtype=int)
        sub = raw[ix]
        out[ix] = (sub / max(sub.sum(), 1e-12)) / len(present)
    out = out * len(records)
    for _ in range(3):
        out = np.minimum(out, 4.0)
        out = out / max(out.mean(), 1e-12)
    return np.minimum(out, 4.0)

def compute_temp_cal_weights(records, mode=CAL_WEIGHT_MODE, that_share=0.70):
    """Trọng số hiệu chuẩn nhiệt độ T: giữ ~70% tổng khối lượng cho câu thật (tránh 57% câu sinh dễ kéo T về 1)
    và 30% cho câu sinh khó (an_y > sinh_doi > lo_lieu) để vẫn đại diện đủ 5 nhãn gồm LUA_DAO."""
    w_base = compute_cal_weights(records, mode=mode)
    if mode != "real_aligned" or not records:
        return w_base
    is_that = np.array([(r.get("meta") or {}).get("loai", "that") == "that" for r in records], dtype=bool)
    if is_that.any() and (~is_that).any():
        out = w_base.copy()
        out[is_that] = out[is_that] / max(out[is_that].sum(), 1e-12) * float(that_share) * len(records)
        out[~is_that] = out[~is_that] / max(out[~is_that].sum(), 1e-12) * (1.0 - float(that_share)) * len(records)
        for _ in range(3):
            out = np.minimum(out, 3.0)
            out = out / max(out.mean(), 1e-12)
        return np.minimum(out, 3.0)
    return w_base

human_cal = read_human(HUMAN_CAL_PATH) if HUMAN_CAL_PATH else []
cal_encoded = [prepare(r) for r in human_cal] if human_cal else encoded["cal"]
cal_records = [e["row"] for e in cal_encoded]
cal = infer(model, cal_encoded)
cal_source = "human" if human_cal else "internal_llm"
cal_ev_weights = np.ones(len(cal_records), dtype=np.float64) if human_cal else compute_cal_weights(cal_records, mode=CAL_WEIGHT_MODE)
cal_temp_weights = np.ones(len(cal_records), dtype=np.float64) if human_cal else compute_temp_cal_weights(cal_records, mode=CAL_WEIGHT_MODE, that_share=0.70)
cal_weights = cal_temp_weights
if not human_cal:
    that_pct = 100.0 * sum(w for r, w in zip(cal_records, cal_temp_weights) if (r.get("meta") or {}).get("loai") == "that") / max(cal_temp_weights.sum(), 1e-12)
    print(f"Hiệu chuẩn trên cal nội bộ ({len(cal_records)} mẫu) [{CAL_WEIGHT_MODE}]: T dùng {that_pct:.1f}% câu thật; bằng chứng cân bằng 5 lớp.")

def arrays(pred):
    return np.array([p["logits"] for p in pred]), np.array([p["gold"] for p in pred])

def _ev_rank_key(st):
    bloat_pen = max(0.0, st["word_length_ratio"] - 1.25) * 0.08
    return (round(st["f1"] - bloat_pen, 4), round(st["precision"], 4), -round(st["safe_mark_rate"], 4))

def _fb_rank_key(st, base_safe_mark):
    bloat_pen = max(0.0, st["word_length_ratio"] - 1.25) * 0.08
    safe_pen = max(0.0, st["safe_mark_rate"] - base_safe_mark) * 0.25
    empty_pen = st["harm_empty_evidence_rate"] * 0.04
    score = st["f1"] - bloat_pen - safe_pen - empty_pen
    return (round(score, 4), round(st["f1_theo_y"], 4), round(st["precision"], 4), -round(st["safe_mark_rate"], 4))


# ==============================================================================
# 2. KHỚP HOẶC NẠP NGUYÊN VẸN CALIBRATION.JSON (NHIỆT ĐỘ T & NGƯỠNG BẰNG CHỨNG)
# ==============================================================================
CAL_PATH = RUN / "calibration.json"
TEST_LOCK_PATH = RUN / "REAL_TEST_LOCK.json"
expected_cal_mode = "human_uniform" if cal_source == "human" else CAL_WEIGHT_MODE
expected_cal_ids = [p["id"] for p in cal]

if TEST_LOCK_PATH.exists() and not CAL_PATH.exists():
    raise AssertionError("CẢNH BÁO: REAL_TEST_LOCK.json đã tồn tại nhưng thiếu calibration.json đã đóng băng — từ chối hiệu chuẩn lại!")
if CAL_PATH.exists() and (EVALUATE_REAL_TEST or TEST_LOCK_PATH.exists()):
    cal_payload = json.loads(CAL_PATH.read_text(encoding="utf8"))
    cal_sha256 = digest(CAL_PATH.read_bytes())
    assert cal_payload.get("ids") == expected_cal_ids, "calibration.json đã đóng băng không khớp danh sách ID tập cal!"
    assert cal_payload.get("source") == cal_source and cal_payload.get("cal_weight_mode") == expected_cal_mode, "calibration.json đã đóng băng lệch nguồn/chế độ hiệu chuẩn!"
    if TEST_LOCK_PATH.exists():
        lock_meta = json.loads(TEST_LOCK_PATH.read_text(encoding="utf8"))
        assert lock_meta.get("frozen_calibration_sha256") == cal_sha256, "CẢNH BÁO: SHA-256 calibration.json không khớp REAL_TEST_LOCK.json!"
    temp = float(cal_payload["temperature"])
    temp_unweighted = float(cal_payload.get("temperature_unweighted", temp))
    base_threshold = float(cal_payload["base_span_threshold"])
    threshold = float(cal_payload["span_threshold"])
    single_word_threshold = float(cal_payload["single_word_threshold"])
    bridge_ratio = float(cal_payload["bridge_ratio"])
    fallback_min_prob = float(cal_payload["fallback_min_prob"])
    print(f"ĐÃ NẠP NGUYÊN VẸN calibration.json ĐÓNG BĂNG (không ghi đè, sha={cal_sha256[:12]}).")
else:
    z, y = arrays(cal)
    def nll_temp(log_t, sample_weight=None):
        return log_loss(y, softmax(z / np.exp(log_t), axis=1), labels=range(len(LABELS)), sample_weight=sample_weight)

    cal_fit_unw = minimize_scalar(lambda lt: nll_temp(lt, None), bounds=(-2.3, 2.3), method="bounded")
    temp_unweighted = float(np.exp(cal_fit_unw.x)) if (cal_fit_unw.success and cal_fit_unw.fun < nll_temp(0.0, None)) else 1.0

    t_bounds = (-2.3, 2.3) if cal_source == "human" else (-0.30, 2.3)
    cal_fit = minimize_scalar(lambda lt: nll_temp(lt, cal_temp_weights), bounds=t_bounds, method="bounded")
    if not cal_fit.success or not np.isfinite([cal_fit.x, cal_fit.fun]).all():
        raise RuntimeError(f"Hiệu chuẩn nhiệt độ thất bại: {cal_fit.message}")
    temp = float(np.exp(cal_fit.x)) if cal_fit.fun < nll_temp(0.0, cal_temp_weights) else 1.0

    if SPAN_WEIGHT:
        coarse = np.linspace(0.15, 0.85, 29)
        t0 = float(max(coarse, key=lambda t: (_ev_rank_key(evidence_stats(cal, t, sample_weights=cal_ev_weights)), -abs(t - 0.5))))
        fine_ts = sorted({round(float(np.clip(t0 + d, 0.125, 0.90)), 3) for d in (-0.05, -0.025, 0.0, 0.025, 0.05)})
        base_threshold = float(max(fine_ts, key=lambda t: (_ev_rank_key(evidence_stats(cal, t, sample_weights=cal_ev_weights)), -abs(t - 0.5))))
        if SPAN_POSTPROC:
            best_span_cfg, best_span_key = None, None
            for t in fine_ts:
                for sw_t in (t, round(min(0.95, t + 0.10), 3), round(min(0.95, t + 0.18), 3), 1.01):
                    for br in (0.0, 0.70, 0.80):
                        st = evidence_stats(cal, t, single_word_threshold=sw_t, bridge_ratio=br, sample_weights=cal_ev_weights)
                        rk = _ev_rank_key(st)
                        if best_span_key is None or rk > best_span_key:
                            best_span_key, best_span_cfg = rk, (t, sw_t, br)
            threshold, single_word_threshold, bridge_ratio = best_span_cfg
            st_nofb = evidence_stats(
                cal, threshold, single_word_threshold=single_word_threshold,
                bridge_ratio=bridge_ratio, fallback_min_prob=1.01, sample_weights=cal_ev_weights,
            )
            fb_grid = (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 1.01)
            fallback_min_prob, best_fb_key = 1.01, None
            for fb in fb_grid:
                st = evidence_stats(
                    cal, threshold, single_word_threshold=single_word_threshold,
                    bridge_ratio=bridge_ratio, fallback_min_prob=fb, sample_weights=cal_ev_weights,
                )
                rk = _fb_rank_key(st, st_nofb["safe_mark_rate"])
                if best_fb_key is None or rk > best_fb_key:
                    best_fb_key, fallback_min_prob = rk, fb
        else:
            threshold, single_word_threshold, bridge_ratio, fallback_min_prob = base_threshold, base_threshold, 0.0, 1.01
    else:
        base_threshold = threshold = single_word_threshold = 1.1
        bridge_ratio, fallback_min_prob = 0.0, 1.01

    cal_payload = dict(
        temperature=temp,
        temperature_unweighted=temp_unweighted,
        cal_weight_mode=expected_cal_mode,
        temp_cal_that_share=round(float(sum(w for r, w in zip(cal_records, cal_temp_weights) if (r.get("meta") or {}).get("loai", "that") == "that") / max(cal_temp_weights.sum(), 1e-12)), 4),
        base_span_threshold=base_threshold,
        span_threshold=threshold,
        single_word_threshold=single_word_threshold,
        bridge_ratio=bridge_ratio,
        fallback_min_prob=fallback_min_prob,
        word_score=WORD_SCORE,
        nll_weighted_before=float(nll_temp(0.0, cal_temp_weights)),
        nll_weighted_after=float(nll_temp(np.log(temp), cal_temp_weights)),
        nll_unweighted_before=float(nll_temp(0.0, None)),
        nll_unweighted_after=float(nll_temp(np.log(temp), None)),
        source=cal_source,
        ids=expected_cal_ids,
    )
    write_json(CAL_PATH, cal_payload)
    cal_sha256 = digest(CAL_PATH.read_bytes())
print(
    f"Hiệu chuẩn [{cal_payload['cal_weight_mode']}]: T={temp:.4f} (unw={temp_unweighted:.4f}) | "
    f"span=(base={base_threshold:.3f}, thr={threshold:.3f}, sw={single_word_threshold:.3f}, br={bridge_ratio}, fb={fallback_min_prob:.2f})"
)


# ==============================================================================
# 3. HÀM ĐÁNH GIÁ TỔNG HỢP (5 NHÃN, LÁT CẮT, BOOTSTRAP 95% CI) & BIỂU ĐỒ
# ==============================================================================
def calibration_stats(prob, y_true):
    confidence = prob.max(1)
    correct = prob.argmax(1) == y_true
    ece = 0.0
    for mask in (np.minimum((confidence * 10).astype(int), 9) == i for i in range(10)):
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return dict(
        nll=float(log_loss(y_true, prob, labels=range(len(LABELS)))),
        ece10=float(ece),
        brier=float(((prob - np.eye(len(LABELS))[y_true]) ** 2).sum(1).mean()),
    )

def format_output(item, text, cal_cfg=None):
    cfg = cal_cfg or cal_payload
    prob = softmax(np.array(item["logits"]) / cfg["temperature"])
    label = int(prob.argmax())
    runs = []
    if label and SPAN_WEIGHT:
        marks = select_word_marks(
            item,
            threshold=cfg["span_threshold"],
            single_word_threshold=cfg["single_word_threshold"],
            bridge_ratio=cfg["bridge_ratio"],
            gate=True,
            fallback_min_prob=cfg["fallback_min_prob"],
        )
        for (a, b, _, _), marked in zip(item["words"], marks):
            if not marked:
                continue
            if runs and re.fullmatch(r"[^\S\n]*", text[runs[-1][1]:a]):
                runs[-1][1] = b
            else:
                runs.append([a, b])
    return dict(
        nhan=LABELS[label],
        tin_cay=float(prob[label]),
        probs=[round(float(x), 6) for x in prob],
        bang_chung=[text[a:b] for a, b in runs],
        vi_tri=runs,
    )

def evaluate(pred, records, cal_cfg=None):
    cfg = cal_cfg or cal_payload
    z_e, y_e = arrays(pred)
    yp = z_e.argmax(1)
    ev_kwargs = dict(
        threshold=cfg["span_threshold"],
        single_word_threshold=cfg["single_word_threshold"],
        bridge_ratio=cfg["bridge_ratio"],
        fallback_min_prob=cfg["fallback_min_prob"],
    )
    safe_mask_e = (y_e == 0)
    result = dict(
        macro_f1=float(macro(pred)),
        accuracy=float((y_e == yp).mean()),
        binary_f1=float(f1_score(y_e != 0, yp != 0, zero_division=0)),
        fpr_an_toan=(float((yp[safe_mask_e] != 0).mean()) if safe_mask_e.any() else 0.0),
        classes=classification_report(y_e, yp, labels=range(5), target_names=LABELS, zero_division=0, output_dict=True),
        confusion=confusion_matrix(y_e, yp, labels=range(5)).tolist(),
        calibration_before=calibration_stats(softmax(z_e, axis=1), y_e),
        calibration_after=calibration_stats(softmax(z_e / cfg["temperature"], axis=1), y_e),
        evidence=evidence_stats(pred, gate=True, **ev_kwargs),
        evidence_single_threshold=evidence_stats(pred, cfg["base_span_threshold"]),
        evidence_without_gate=evidence_stats(pred, gate=False, **ev_kwargs),
    )
    that_ix = [i for i, r in enumerate(records) if (r.get("meta") or {}).get("loai") == "that"]
    if that_ix:
        result["calibration_real_before"] = calibration_stats(softmax(z_e[that_ix], axis=1), y_e[that_ix])
        result["calibration_real_after"] = calibration_stats(softmax(z_e[that_ix] / cfg["temperature"], axis=1), y_e[that_ix])

    slices = defaultdict(list)
    for i, r in enumerate(records):
        meta = r.get("meta") or {}
        loai = meta.get("loai") or r.get("nguon", "khac")
        keys = [loai, "1 lượt" if r["so_dong"] == 1 else "nhiều lượt", *r.get("the", [])]
        phan_t = r.get("phan_test") or meta.get("phan_test")
        if phan_t:
            keys.append("phan:" + phan_t)
        vung = r.get("giong_vung") or meta.get("giong_vung")
        if not vung and meta.get("giong") in ("giong_bac", "giong_trung", "giong_nam"):
            vung = meta["giong"].removeprefix("giong_")
        if vung:
            keys.append("vung:" + vung)
        do_lo = r.get("do_lo") or meta.get("do_lo")
        if do_lo:
            keys.append("do_lo:" + do_lo)
        kenh = r.get("kenh") or meta.get("kenh")
        if kenh:
            keys.append("kenh:" + kenh)
        if r.get("cach_gan"):
            keys.append("cach_gan:" + r["cach_gan"])
        bd = r.get("bien_dang") or meta.get("bien_dang")
        if bd:
            keys.append("bien_dang:" + bd)
        nbd = r.get("nhom_bien_dang") or meta.get("nhom_bien_dang")
        if nbd:
            keys.append("nhom_bd:" + nbd)
        kbd = r.get("kieu_bien_dang") or meta.get("kieu_bien_dang")
        if kbd:
            keys.append("kieu_bd:" + kbd)
        for key in keys:
            slices[key].append(i)
    result["slices"] = {}
    for key, ix in slices.items():
        present = np.unique(y_e[ix])
        sm_ix = (y_e[ix] == 0)
        result["slices"][key] = dict(
            n=len(ix),
            labels=[LABELS[j] for j in present],
            macro_present=float(f1_score(y_e[ix], yp[ix], average="macro", labels=present, zero_division=0)),
            binary_f1=float(f1_score(y_e[ix] != 0, yp[ix] != 0, zero_division=0)),
            fpr_an_toan=(float((yp[ix][sm_ix] != 0).mean()) if sm_ix.any() else 0.0),
            accuracy=float((y_e[ix] == yp[ix]).mean()),
            evidence=evidence_stats([pred[i] for i in ix], gate=True, **ev_kwargs),
        )
    reg_f1s = [result["slices"][f"vung:{g}"]["macro_present"] for g in ("bac", "trung", "nam") if f"vung:{g}" in result["slices"]]
    if len(reg_f1s) == 3:
        result["worst_group_region_macro_f1"] = float(min(reg_f1s))
        result["region_gap_max_min"] = float(max(reg_f1s) - min(reg_f1s))
    result["recall_nguon_va_nhan"] = {}
    for source in sorted({(r.get("meta") or {}).get("loai") or r.get("nguon", "khac") for r in records}):
        table = {}
        for j, label in enumerate(LABELS):
            ix = [i for i, r in enumerate(records) if ((r.get("meta") or {}).get("loai") or r.get("nguon", "khac")) == source and y_e[i] == j]
            table[label] = {"n": len(ix), "recall": float((yp[ix] == j).mean()) if ix else None}
        result["recall_nguon_va_nhan"][source] = table

    main4 = [LABELS.index(l) for l in ("AN_TOAN", "QUAY_ROI", "DE_DOA", "LUA_DAO")]
    result["macro_f1_4main"] = float(f1_score(y_e, yp, labels=main4, average="macro", zero_division=0))

    group_names, inverse = np.unique([split_group(r) for r in records], return_inverse=True)
    matrices = np.zeros((len(group_names), len(LABELS), len(LABELS)), dtype=np.int64)
    np.add.at(matrices, (inverse, y_e, yp), 1)
    rng = np.random.default_rng(SEED)
    scores, scores_4, scores_bin = [], [], []
    for _ in range(500):
        cm = matrices[rng.integers(0, len(matrices), len(matrices))].sum(0)
        per_c = 2 * cm.diagonal() / np.maximum(cm.sum(0) + cm.sum(1), 1)
        scores.append(per_c.mean())
        scores_4.append(per_c[main4].mean())
        tp_b = cm[1:, 1:].sum()
        fp_b = cm[0, 1:].sum()
        fn_b = cm[1:, 0].sum()
        scores_bin.append(2 * tp_b / max(2 * tp_b + fp_b + fn_b, 1))
    result["macro_f1_bootstrap_95"] = np.quantile(scores, [0.025, 0.975]).tolist()
    result["macro_f1_4main_bootstrap_95"] = np.quantile(scores_4, [0.025, 0.975]).tolist()
    result["binary_f1_bootstrap_95"] = np.quantile(scores_bin, [0.025, 0.975]).tolist()
    return result

def save_diagnostics_plot(pred, out_path, cal_cfg=None):
    cfg = cal_cfg or cal_payload
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), layout="constrained")
    z_eval, y_eval = arrays(pred)
    ConfusionMatrixDisplay.from_predictions(
        y_eval, z_eval.argmax(1), labels=range(5), display_labels=LABELS,
        xticks_rotation=40, ax=axes[0], colorbar=False, cmap="Blues",
    )
    prob_raw = softmax(z_eval, axis=1)
    prob_eval = softmax(z_eval / cfg["temperature"], axis=1)
    for j, name in enumerate(LABELS[1:], start=1):
        target = y_eval == j
        if target.any():
            precision, recall, _ = precision_recall_curve(target, prob_eval[:, j])
            ap = average_precision_score(target, prob_eval[:, j])
            axes[1].plot(recall, precision, label=f"{name} · AP {ap:.2f}")
    axes[1].set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02), title="Precision–recall từng nhãn nguy hại")
    axes[1].legend(fontsize=8)

    def _rel_curve(prob):
        conf, corr = prob.max(1), prob.argmax(1) == y_eval
        xs, ys = [], []
        for i in range(10):
            m = np.minimum((conf * 10).astype(int), 9) == i
            if m.any():
                xs.append(float(conf[m].mean()))
                ys.append(float(corr[m].mean()))
        return xs, ys

    ece_before = calibration_stats(prob_raw, y_eval)["ece10"]
    ece_after = calibration_stats(prob_eval, y_eval)["ece10"]
    axes[2].plot([0, 1], [0, 1], "k--", lw=1, label="Hoàn hảo")
    axes[2].plot(*_rel_curve(prob_raw), "o-", label=f"Trước (ECE {ece_before:.3f})")
    axes[2].plot(*_rel_curve(prob_eval), "s-", label=f"Sau T={cfg['temperature']:.2f} (ECE {ece_after:.3f})")
    axes[2].set(xlabel="Độ tin cậy", ylabel="Độ chính xác", xlim=(0, 1), ylim=(0, 1.02), title="Độ tin cậy (Reliability)")
    axes[2].legend(fontsize=8)
    fig.savefig(out_path, dpi=140)
    plt.show()
    plt.close(fig)


# ==============================================================================
# 4. XUẤT BÁO CÁO TẬP PHÁT TRIỂN (DEV / HUMAN_CAL / AUDIT)
# ==============================================================================
def report_split(key, data_override=None):
    data = encoded[key] if data_override is None else data_override
    records = [e["row"] for e in data]
    pred = infer(model, data)
    result = evaluate(pred, records)
    if key in ("cal", "human_cal"):
        result["diagnostic_split_warning"] = "in_sample_cal_only_do_not_use_for_model_selection"
        print(f"LƯU Ý ({DATA_SCHEME}): Đang xuất chẩn đoán trên {key}; chỉ dùng để kiểm tra ống suy luận, không dùng chọn mô hình.")
    write_json(RUN / f"{key}_metrics.json", result)
    with (RUN / f"{key}_predictions.jsonl").open("w", encoding="utf8") as f:
        for p, r in zip(pred, records):
            record = dict(id=r["id"], nhan_dung=r["nhan"], nguon=r["nguon"], **format_output(p, r["text"]))
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(
        key, "macro-F1", round(result["macro_f1"], 4), "95%", [round(x, 4) for x in result["macro_f1_bootstrap_95"]],
        "| Ev F1 (word / theo_y):", round(result["evidence"]["f1"], 4), "/", round(result["evidence"]["f1_theo_y"], 4),
        "| len_ratio:", round(result["evidence"]["word_length_ratio"], 2),
        "| empty_harm:", f"{100*result['evidence']['harm_empty_evidence_rate']:.2f}%",
    )
    print(pd.DataFrame(result["confusion"], index=LABELS, columns=LABELS))
    print(pd.DataFrame(result["classes"]).T)
    print(pd.DataFrame(result["slices"]).T[["n", "macro_present", "accuracy"]])
    return pred

if EVALUATE_AUDIT and "audit" not in encoded:
    print(f"CẢNH BÁO: EVALUATE_AUDIT=True nhưng DATA_SCHEME={DATA_SCHEME!r} không có split audit; bỏ qua audit.")
if EVALUATE_AUDIT and "audit" in encoded:
    report_key, report_encoded = "audit", encoded["audit"]
elif "dev" in encoded:
    report_key, report_encoded = "dev", encoded["dev"]
elif "cal" in encoded:
    report_key, report_encoded = "cal", encoded["cal"]
else:
    report_key, report_encoded = "human_cal", cal_encoded
final_pred = report_split(report_key, data_override=report_encoded)
save_diagnostics_plot(final_pred, RUN / f"{report_key}_diagnostics.png")

# ==============================================================================
# 5. KHÓA CỨNG BỘ TEST ĐÁNH GIÁ (334 C0 + 668 C1 + 668 C2 = 1.670 MẪU, #5 & #6)
# ==============================================================================
def load_locked_real_test():
    test_full = fetch_data("test_day_du.jsonl")
    test_slim = fetch_data("test.jsonl")
    assert len(test_full) == len(test_slim) == 334
    train_ids = {r["id"] for r in rows}
    train_texts = {r["text"].strip().casefold() for r in rows}
    train_norms = {nt for nt in (_norm_dedup_text(r["text"]) for r in rows) if nt}
    hcal_ids = {r["id"] for r in human_cal}
    hcal_texts = {r["text"].strip().casefold() for r in human_cal}
    hcal_norms = {nt for nt in (_norm_dedup_text(r["text"]) for r in human_cal) if nt}
    assert len({r["id"] for r in test_full}) == len(test_full)
    for r, s in zip(test_full, test_slim):
        assert s == {k: r[k] for k in ("id", "text", "nhan", "bang_chung")}
        nt = _norm_dedup_text(r["text"])
        assert r["id"] not in train_ids and r["text"].strip().casefold() not in train_texts and (not nt or nt not in train_norms)
        assert r["id"] not in hcal_ids and r["text"].strip().casefold() not in hcal_texts and (not nt or nt not in hcal_norms)
        assert r.get("nguon") != "sinh_llm" and r.get("cach_gan") != "llm"
        assert r["nhan"] in LABELS
        assert r["nhan"] != "AN_TOAN" or not r["bang_chung"]
        lines = r["text"].split("\n")
        assert 1 <= len(lines) <= 8 and r["so_dong"] == len(lines)
        assert all(p and "\n" not in p and any(p in ln for ln in lines) for p in r["bang_chung"])
        if not r.get("nhom_trung"):
            r["nhom_trung"] = r["id"]
        r.setdefault("bien_dang", "C0")
        r.setdefault("goc_id", r["id"])
        r["meta"] = {"loai": "that", "phan_test": r.get("phan_test", ""), "giong_vung": r.get("giong_vung", ""), "bien_dang": "C0"}
    return test_full

def load_locked_real_test_perturb(test_c0):
    c0_by_id = {r["id"]: r for r in test_c0}
    train_ids = {r["id"] for r in rows}
    hcal_ids = {r["id"] for r in human_cal}
    seen_ids = set(c0_by_id)
    out = {}
    for tier, fname in (("C1", "test_perturb_C1.jsonl"), ("C2", "test_perturb_C2.jsonl")):
        recs = fetch_data(fname)
        assert len(recs) == 668, f"{fname}: kỳ vọng 668 dòng, nhận {len(recs)}"
        enriched = []
        for r in recs:
            rid = r["id"]
            assert rid not in seen_ids and rid not in train_ids and rid not in hcal_ids, f"Trùng/rò rỉ id {rid} trong {fname}"
            seen_ids.add(rid)
            gid = r["goc_id"]
            assert gid in c0_by_id, f"Thiếu goc_id {gid} cho {rid}"
            src = c0_by_id[gid]
            assert r["nhan"] == src["nhan"] and r["nhan"] in LABELS and r.get("bien_dang") == tier
            assert r["nhan"] != "AN_TOAN" or not r["bang_chung"]
            lines = r["text"].split("\n")
            assert 1 <= len(lines) <= 8
            assert all(p and "\n" not in p and any(p in ln for ln in lines) for p in r["bang_chung"])
            rec = dict(r)
            rec["so_dong"] = len(lines)
            rec["nhom_trung"] = gid
            for k in ("nguon", "cach_gan", "do_lo", "kenh", "the"):
                if k not in rec and k in src:
                    rec[k] = src[k]
            rec["meta"] = {
                "loai": "that",
                "phan_test": rec.get("phan_test", ""),
                "giong_vung": rec.get("giong_vung", ""),
                "bien_dang": tier,
                "kieu_bien_dang": rec.get("kieu_bien_dang", ""),
                "nhom_bien_dang": rec.get("nhom_bien_dang", ""),
            }
            enriched.append(rec)
        out[tier] = enriched
    return out["C1"], out["C2"]

def _compact_eval_summary(m, n):
    out = dict(
        n=int(n),
        macro_f1=round(float(m["macro_f1"]), 4),
        macro_f1_4main=round(float(m["macro_f1_4main"]), 4),
        binary_f1=round(float(m["binary_f1"]), 4),
        fpr_an_toan=round(float(m["fpr_an_toan"]), 4),
        accuracy=round(float(m["accuracy"]), 4),
        macro_f1_bootstrap_95=[round(float(x), 4) for x in m["macro_f1_bootstrap_95"]],
        evidence_f1=round(float(m["evidence"]["f1"]), 4),
        evidence_f1_theo_y=round(float(m["evidence"]["f1_theo_y"]), 4),
        safe_mark_rate=round(float(m["evidence"]["safe_mark_rate"]), 4),
        word_length_ratio=round(float(m["evidence"]["word_length_ratio"]), 4),
        harm_empty_evidence_rate=round(float(m["evidence"]["harm_empty_evidence_rate"]), 4),
    )
    if "worst_group_region_macro_f1" in m:
        out["worst_group_region_macro_f1"] = round(float(m["worst_group_region_macro_f1"]), 4)
        out["region_gap_max_min"] = round(float(m["region_gap_max_min"]), 4)
    return out

def _kieu_seen_in_train_aug(kieu):
    if kieu == "bo_dau":
        return bool(ACCENT_AUG > 0 or SURFACE_AUG > 0)
    if kieu in ("teencode", "keo_dai", "chen_ky_tu", "so_thay_chu"):
        return bool(SURFACE_AUG > 0)
    return False

def build_official_test_matrix(c0_rec, c0_pred, c1_rec, c1_pred, c2_rec, c2_pred, cal_cfg, raw_matrix=None):
    m0 = evaluate(c0_pred, c0_rec, cal_cfg=cal_cfg)
    m1 = evaluate(c1_pred, c1_rec, cal_cfg=cal_cfg)
    m2 = evaluate(c2_pred, c2_rec, cal_cfg=cal_cfg)
    tiers = {"C0": _compact_eval_summary(m0, len(c0_rec)), "C1": _compact_eval_summary(m1, len(c1_rec)), "C2": _compact_eval_summary(m2, len(c2_rec))}
    deltas = dict(
        C1_minus_C0=round(tiers["C1"]["macro_f1"] - tiers["C0"]["macro_f1"], 4),
        C2_minus_C0=round(tiers["C2"]["macro_f1"] - tiers["C0"]["macro_f1"], 4),
        C1_minus_C0_binary=round(tiers["C1"]["binary_f1"] - tiers["C0"]["binary_f1"], 4),
        C2_minus_C0_binary=round(tiers["C2"]["binary_f1"] - tiers["C0"]["binary_f1"], 4),
        C1_minus_C0_ev_theo_y=round(tiers["C1"]["evidence_f1_theo_y"] - tiers["C0"]["evidence_f1_theo_y"], 4),
        C2_minus_C0_ev_theo_y=round(tiers["C2"]["evidence_f1_theo_y"] - tiers["C0"]["evidence_f1_theo_y"], 4),
    )
    slices_15 = {}
    for tier_name, m in (("C0", m0), ("C1", m1), ("C2", m2)):
        for src_k, dst_k in (
            ("vung:bac", "bac"), ("vung:trung", "trung"), ("vung:nam", "nam"),
            ("phan:trong_phan_phoi", "da_thay"), ("phan:chua_thay", "chua_thay"),
        ):
            if src_k in m["slices"]:
                sl = m["slices"][src_k]
                slices_15[f"{tier_name}_{dst_k}"] = dict(
                    n=sl["n"],
                    macro_f1=round(float(sl["macro_present"]), 4),
                    binary_f1=round(float(sl["binary_f1"]), 4),
                    fpr_an_toan=round(float(sl["fpr_an_toan"]), 4),
                    evidence_f1=round(float(sl["evidence"]["f1"]), 4),
                    evidence_f1_theo_y=round(float(sl["evidence"]["f1_theo_y"]), 4),
                )
    delta_slices = {}
    for grp in ("bac", "trung", "nam", "da_thay", "chua_thay"):
        if f"C0_{grp}" in slices_15:
            for tier_name in ("C1", "C2"):
                if f"{tier_name}_{grp}" in slices_15:
                    delta_slices[f"{tier_name}_{grp}_minus_C0"] = round(
                        slices_15[f"{tier_name}_{grp}"]["macro_f1"] - slices_15[f"C0_{grp}"]["macro_f1"], 4
                    )
    by_kieu = {}
    for tier_name, m, recs in (("C1", m1, c1_rec), ("C2", m2, c2_rec)):
        kieu_nhom = {r.get("kieu_bien_dang", ""): r.get("nhom_bien_dang", "") for r in recs}
        for k, sl in sorted(m["slices"].items()):
            if k.startswith("kieu_bd:"):
                kieu = k.removeprefix("kieu_bd:")
                nhom_c0 = kieu_nhom.get(kieu, "")
                seen_aug = _kieu_seen_in_train_aug(kieu)
                by_kieu[kieu] = dict(
                    tier=tier_name,
                    n=sl["n"],
                    nhom_vs_train_c0=nhom_c0,
                    seen_in_train_aug=seen_aug,
                    strictly_unseen=(nhom_c0 == "chua_thay" and not seen_aug),
                    macro_f1=round(float(sl["macro_present"]), 4),
                    binary_f1=round(float(sl["binary_f1"]), 4),
                    fpr_an_toan=round(float(sl["fpr_an_toan"]), 4),
                    evidence_f1=round(float(sl["evidence"]["f1"]), 4),
                    evidence_f1_theo_y=round(float(sl["evidence"]["f1_theo_y"]), 4),
                )
    unseen_recs, unseen_preds = [], []
    for r, p in list(zip(c1_rec, c1_pred)) + list(zip(c2_rec, c2_pred)):
        if r.get("nhom_bien_dang") == "chua_thay" and not _kieu_seen_in_train_aug(r.get("kieu_bien_dang", "")):
            unseen_recs.append(r)
            unseen_preds.append(p)
    strictly_unseen = _compact_eval_summary(evaluate(unseen_preds, unseen_recs, cal_cfg=cal_cfg), len(unseen_recs)) if unseen_recs else None
    matrix = dict(
        data_rev=DATA_REV,
        train_aug_config=dict(accent_aug=ACCENT_AUG, surface_aug=SURFACE_AUG),
        tiers=tiers,
        deltas_vs_C0=deltas,
        strictly_unseen_perturb=strictly_unseen,
        slices_15=slices_15,
        delta_slices_vs_C0=delta_slices,
        by_kieu_bien_dang=by_kieu,
    )
    if raw_matrix is not None:
        gain = {t: round(tiers[t]["macro_f1"] - raw_matrix["tiers"][t]["macro_f1"], 4) for t in ("C0", "C1", "C2")}
        gain["C1_ev_theo_y"] = round(tiers["C1"]["evidence_f1_theo_y"] - raw_matrix["tiers"]["C1"]["evidence_f1_theo_y"], 4)
        gain["C2_ev_theo_y"] = round(tiers["C2"]["evidence_f1_theo_y"] - raw_matrix["tiers"]["C2"]["evidence_f1_theo_y"], 4)
        if strictly_unseen and raw_matrix.get("strictly_unseen_perturb"):
            gain["strictly_unseen"] = round(strictly_unseen["macro_f1"] - raw_matrix["strictly_unseen_perturb"]["macro_f1"], 4)
        gain["by_kieu"] = {
            k: round(by_kieu[k]["macro_f1"] - raw_matrix["by_kieu_bien_dang"][k]["macro_f1"], 4)
            for k in by_kieu if k in raw_matrix["by_kieu_bien_dang"]
        }
        matrix["gain_vs_raw"] = gain
    return matrix, (m0, m1, m2)

def _write_real_test_jsonl(pred_path, err_path, pred_list, rec_list, cal_cfg):
    rows_out, errors_out = [], []
    with pred_path.open("w", encoding="utf8") as f:
        for p, r in zip(pred_list, rec_list):
            out_rec = format_output(p, r["text"], cal_cfg=cal_cfg)
            row_out = dict(
                id=r["id"], nhan_dung=r["nhan"], nguon=r.get("nguon", "nguoi_viet"), cach_gan=r.get("cach_gan", ""),
                bien_dang=r.get("bien_dang", "C0"), goc_id=r.get("goc_id", r["id"]),
                kieu_bien_dang=r.get("kieu_bien_dang", "goc"), nhom_bien_dang=r.get("nhom_bien_dang", "goc"),
                phan_test=r.get("phan_test", ""), giong_vung=r.get("giong_vung", ""),
                kich_ban=r.get("kich_ban", ""), do_lo=r.get("do_lo", ""), kenh=r.get("kenh", ""),
                **out_rec,
            )
            f.write(json.dumps(row_out, ensure_ascii=False) + "\n")
            rows_out.append(row_out)
            if out_rec["nhan"] != r["nhan"]:
                errors_out.append(dict(row_out, text=r["text"], bang_chung_dung=r["bang_chung"]))
    if err_path is not None:
        with err_path.open("w", encoding="utf8") as f:
            for err in errors_out:
                f.write(json.dumps(err, ensure_ascii=False) + "\n")
    return rows_out, errors_out

real_test_records = real_test_encoded = real_test_pred = None
real_test_c1_records = real_test_c2_records = real_test_c1_pred = real_test_c2_pred = None
TEENCODE_DICT_MAP = fetch_teencode_dict() if EVAL_WITH_NORMALIZER else {}

if EVALUATE_REAL_TEST:
    if RUN_MODE != "full":
        raise RuntimeError("KHÓA AN TOÀN: Không chấm tập test thật ở RUN_MODE='smoke'! Đổi RUN_MODE='full' trước.")
    if TEST_LOCK_PATH.exists():
        lock_info = json.loads(TEST_LOCK_PATH.read_text(encoding="utf8"))
        print(f"ĐÃ KHÓA (1-SHOT LOCK): {TEST_LOCK_PATH} đã tồn tại (locked_at={lock_info.get('locked_at_utc')}, macro_f1={lock_info.get('macro_f1'):.4f}) — từ chối chấm lại lần 2!")
    else:
        frozen_cal = json.loads(CAL_PATH.read_text(encoding="utf8"))
        frozen_cal_sha = digest(CAL_PATH.read_bytes())
        real_test_records = load_locked_real_test()
        real_test_encoded = [prepare(r) for r in real_test_records]
        real_test_pred = infer(model, real_test_encoded)
        real_test_metrics = evaluate(real_test_pred, real_test_records, cal_cfg=frozen_cal)
        real_test_metrics["test_dataset_rev"] = DATA_REV
        real_test_metrics["test_day_du_sha256"] = DATA_SHA256["test_day_du.jsonl"]
        real_test_metrics["frozen_calibration_sha256"] = frozen_cal_sha
        write_json(RUN / "real_test_metrics.json", real_test_metrics)
        c0_rows_out, errors_out = _write_real_test_jsonl(
            RUN / "real_test_predictions.jsonl", RUN / "real_test_errors.jsonl",
            real_test_pred, real_test_records, frozen_cal,
        )
        save_diagnostics_plot(real_test_pred, RUN / "real_test_diagnostics.png", cal_cfg=frozen_cal)
        lock_payload = dict(
            locked_at_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            data_rev=DATA_REV,
            test_day_du_sha256=DATA_SHA256["test_day_du.jsonl"],
            test_perturb_c1_sha256=(DATA_SHA256["test_perturb_C1.jsonl"] if RUN_REAL_TEST_PERTURB else None),
            test_perturb_c2_sha256=(DATA_SHA256["test_perturb_C2.jsonl"] if RUN_REAL_TEST_PERTURB else None),
            frozen_calibration_sha256=frozen_cal_sha,
            n_test=len(real_test_records),
            n_c1=0,
            n_c2=0,
            macro_f1=real_test_metrics["macro_f1"],
            macro_f1_4main=real_test_metrics["macro_f1_4main"],
            binary_f1=real_test_metrics["binary_f1"],
            fpr_an_toan=real_test_metrics["fpr_an_toan"],
            evidence_f1=real_test_metrics["evidence"]["f1"],
            evidence_f1_theo_y=real_test_metrics["evidence"]["f1_theo_y"],
            n_errors=len(errors_out),
        )
        raw_matrix = None
        if RUN_REAL_TEST_PERTURB:
            real_test_c1_records, real_test_c2_records = load_locked_real_test_perturb(real_test_records)
            real_test_c1_pred = infer(model, [prepare(r) for r in real_test_c1_records])
            real_test_c2_pred = infer(model, [prepare(r) for r in real_test_c2_records])
            raw_matrix, (_, m_c1, m_c2) = build_official_test_matrix(
                real_test_records, real_test_pred,
                real_test_c1_records, real_test_c1_pred,
                real_test_c2_records, real_test_c2_pred,
                frozen_cal,
            )
            write_json(RUN / "real_test_c1_metrics.json", m_c1)
            write_json(RUN / "real_test_c2_metrics.json", m_c2)
            write_json(RUN / "real_test_official_matrix.json", raw_matrix)
            c1_rows_out, _ = _write_real_test_jsonl(
                RUN / "real_test_c1_predictions.jsonl", RUN / "real_test_c1_errors.jsonl",
                real_test_c1_pred, real_test_c1_records, frozen_cal,
            )
            c2_rows_out, _ = _write_real_test_jsonl(
                RUN / "real_test_c2_predictions.jsonl", RUN / "real_test_c2_errors.jsonl",
                real_test_c2_pred, real_test_c2_records, frozen_cal,
            )
            with (RUN / "real_test_all_predictions.jsonl").open("w", encoding="utf8") as f:
                for row_out in c0_rows_out + c1_rows_out + c2_rows_out:
                    f.write(json.dumps(row_out, ensure_ascii=False) + "\n")
            lock_payload.update(
                n_c1=len(real_test_c1_records),
                n_c2=len(real_test_c2_records),
                c1_macro_f1=raw_matrix["tiers"]["C1"]["macro_f1"],
                c2_macro_f1=raw_matrix["tiers"]["C2"]["macro_f1"],
                delta_c1_minus_c0=raw_matrix["deltas_vs_C0"]["C1_minus_C0"],
                delta_c2_minus_c0=raw_matrix["deltas_vs_C0"]["C2_minus_C0"],
                c1_evidence_f1_theo_y=raw_matrix["tiers"]["C1"]["evidence_f1_theo_y"],
                c2_evidence_f1_theo_y=raw_matrix["tiers"]["C2"]["evidence_f1_theo_y"],
            )
        if EVAL_WITH_NORMALIZER:
            rt_norm_c0_pred = infer_normalized(model, real_test_records, TEENCODE_DICT_MAP)
            nc0_rows_out, _ = _write_real_test_jsonl(
                RUN / "real_test_norm_predictions.jsonl", None,
                rt_norm_c0_pred, real_test_records, frozen_cal,
            )
            if RUN_REAL_TEST_PERTURB:
                rt_norm_c1_pred = infer_normalized(model, real_test_c1_records, TEENCODE_DICT_MAP)
                rt_norm_c2_pred = infer_normalized(model, real_test_c2_records, TEENCODE_DICT_MAP)
                norm_matrix, (m0_norm, m1_norm, m2_norm) = build_official_test_matrix(
                    real_test_records, rt_norm_c0_pred,
                    real_test_c1_records, rt_norm_c1_pred,
                    real_test_c2_records, rt_norm_c2_pred,
                    frozen_cal, raw_matrix=raw_matrix,
                )
                write_json(RUN / "real_test_norm_metrics.json", m0_norm)
                write_json(RUN / "real_test_norm_c1_metrics.json", m1_norm)
                write_json(RUN / "real_test_norm_c2_metrics.json", m2_norm)
                write_json(RUN / "real_test_norm_official_matrix.json", norm_matrix)
                nc1_rows_out, _ = _write_real_test_jsonl(
                    RUN / "real_test_norm_c1_predictions.jsonl", None,
                    rt_norm_c1_pred, real_test_c1_records, frozen_cal,
                )
                nc2_rows_out, _ = _write_real_test_jsonl(
                    RUN / "real_test_norm_c2_predictions.jsonl", None,
                    rt_norm_c2_pred, real_test_c2_records, frozen_cal,
                )
                with (RUN / "real_test_norm_all_predictions.jsonl").open("w", encoding="utf8") as f:
                    for row_out in nc0_rows_out + nc1_rows_out + nc2_rows_out:
                        f.write(json.dumps(row_out, ensure_ascii=False) + "\n")
                lock_payload.update(
                    norm_c0_macro_f1=norm_matrix["tiers"]["C0"]["macro_f1"],
                    norm_c1_macro_f1=norm_matrix["tiers"]["C1"]["macro_f1"],
                    norm_c2_macro_f1=norm_matrix["tiers"]["C2"]["macro_f1"],
                    norm_gain_c1=norm_matrix["gain_vs_raw"]["C1"],
                )
            else:
                norm_c0_m = evaluate(rt_norm_c0_pred, real_test_records, cal_cfg=frozen_cal)
                write_json(RUN / "real_test_norm_metrics.json", norm_c0_m)
                lock_payload["norm_c0_macro_f1"] = round(float(norm_c0_m["macro_f1"]), 4)
        write_json(TEST_LOCK_PATH, lock_payload)
        print(
            f"=== KẾT QUẢ TEST THẬT ĐÃ KHÓA (C0={len(real_test_records)}, C1={lock_payload['n_c1']}, C2={lock_payload['n_c2']}) ===\n"
            f"C0 macro-F1 (5 lớp): {real_test_metrics['macro_f1']:.4f} | FPR(AN_TOAN): {real_test_metrics['fpr_an_toan']:.4f} | "
            f"binary-F1: {real_test_metrics['binary_f1']:.4f} | Ev F1 (word/theo_y): "
            f"{real_test_metrics['evidence']['f1']:.4f} / {real_test_metrics['evidence']['f1_theo_y']:.4f} | "
            f"safe_mark: {real_test_metrics['evidence']['safe_mark_rate']:.4f} | empty_harm: {100*real_test_metrics['evidence']['harm_empty_evidence_rate']:.2f}%"
        )
        if raw_matrix is not None:
            print("Ma trận chính thức C0 / C1 / C2 (Dòng #5 thô):", raw_matrix["tiers"])
            print("Độ tụt ΔF1 so với C0:", raw_matrix["deltas_vs_C0"])
        if EVAL_WITH_NORMALIZER and RUN_REAL_TEST_PERTURB:
            print("Lợi ích chuẩn hóa Dòng #6 − #5:", norm_matrix["gain_vs_raw"])
        print(pd.DataFrame(real_test_metrics["slices"]).T[["n", "macro_present", "binary_f1", "accuracy"]])
else:
    print("Bộ test thật (334 C0 + 668 C1 + 668 C2 = 1.670 mẫu) đang ĐÓNG KHÓA (EVALUATE_REAL_TEST = False).")

---
<a id="sec-7"></a>
## 7. Kiểm thử độ bền (Stress Tests), Ngữ cảnh đa lượt & Benchmark hiệu năng

Ô này thực hiện kiểm thử độ bền biến dạng (gắn cờ `seen_in_train` tường minh cho từng bộ), kiểm chứng đóng góp của ngữ cảnh hội thoại và đo chi phí suy luận sạch trên GPU:

| Nhóm kiểm thử | Bộ phép thử / Chỉ số | Trạng thái lúc huấn luyện |
| :--- | :--- | :---: |
| **Stress Tests (Đã thấy nếu bật Aug)** | `accent_strip` (khi `ACCENT_AUG > 0`), `c1_in_dist` (khi `SURFACE_AUG > 0`) — tuyệt đối không báo cáo là "chưa thấy" | `seen_in_train = True` (khi bật) |
| **Stress Tests (Chưa thấy)** | `c1_heldout_controlled` (cụm phương ngữ/tiếng lóng trong `controlled.py`, lookalike `s/g/b`, dấu tách `·`, lỗi gõ QWERTY), `blindspot_homoglyph`, `blindspot_zero_width` | **Chưa thấy (`seen_in_train = False`)** |
| **Xem trước Chuẩn hóa `#6`** | `{report_key}_norm_stress_tests.json` (`#6 − #5` trên `clean_c0` và 5 bộ stress trên `dev`) | Không chạm bộ test khóa |
| **Kiểm chứng ngữ cảnh** | Bộ phân loại cây chỉ dùng độ dài (Length Baseline) & So sánh giữ đủ hội thoại vs chỉ giữ lượt cuối | `{report_key}_context_checks.json` |
| **Benchmark & Demo** | Đo VRAM sạch, độ trễ `p50/p95 ms` (`benchmark.json`) và chạy thử `predict_text(..., use_normalizer=...)` | `benchmark.json` |

In [ ]:
#@title 7.1. Kiểm thử độ bền (5 bộ + #6), Bỏ ngữ cảnh, Đo VRAM/Độ trễ & Demo predict_text
# ==============================================================================
# 1. ĐỊNH NGHĨA 5 BỘ KIỂM THỬ BIẾN DẠNG TẤT ĐỊNH
# ==============================================================================
def _stress_eval(label, transform_fn, seen_in_train, eval_encoded=None, eval_pred=None):
    eval_encoded = report_encoded if eval_encoded is None else eval_encoded
    eval_pred = final_pred if eval_pred is None else eval_pred
    changed, original = [], []
    for e, p in zip(eval_encoded, eval_pred):
        r = e["row"]
        variant = transform_fn(r["text"])
        if variant != r["text"]:
            changed.append(prepare(dict(r, text=variant, bang_chung=[])))
            original.append(p)
    if not changed:
        return None
    stress = infer(model, changed)
    a, truth = arrays(original)
    b, _ = arrays(stress)
    orig_f1, pert_f1 = float(macro(original)), float(macro(stress))
    main4 = [LABELS.index(l) for l in ("AN_TOAN", "QUAY_ROI", "DE_DOA", "LUA_DAO")]
    orig_f1_4 = float(f1_score(truth, a.argmax(1), labels=main4, average="macro", zero_division=0))
    pert_f1_4 = float(f1_score(truth, b.argmax(1), labels=main4, average="macro", zero_division=0))
    orig_bin = float(f1_score(truth != 0, a.argmax(1) != 0, zero_division=0))
    pert_bin = float(f1_score(truth != 0, b.argmax(1) != 0, zero_division=0))
    return dict(
        suite=label,
        seen_in_train=bool(seen_in_train),
        n=len(stress),
        original_f1=orig_f1,
        perturbed_f1=pert_f1,
        delta_f1=pert_f1 - orig_f1,
        original_f1_4main=orig_f1_4,
        perturbed_f1_4main=pert_f1_4,
        original_binary_f1=orig_bin,
        perturbed_binary_f1=pert_bin,
        flip_rate=float((a.argmax(1) != b.argmax(1)).mean()),
        newly_missed_harm=int(((truth != 0) & (a.argmax(1) != 0) & (b.argmax(1) == 0)).sum()),
    )

def _sample_rng(offset, text):
    return random.Random(f"{SEED}:{offset}:{text}")

def _eval_c1_in_dist(text):
    rng = _sample_rng(77, text)
    lines = []
    for line in text.split("\n"):
        row_stub = augment_surface(dict(id="eval", text=line, nhan="AN_TOAN", bang_chung=[]), rng)
        lines.append(row_stub["text"])
    return "\n".join(lines)

# Bộ quy tắc C1 tách rời hoàn toàn khỏi augment_surface (khớp src/variant_generator/controlled.py)
_HELDOUT_PHRASES = {
    "bây giờ": ("bg", "bi h"),
    "bao giờ": ("bh",),
    "như thế nào": ("ntn",),
    "được không": ("đc ko",),
    "tại sao": ("tsao",),
    "anh em": ("ae",),
    "bình thường": ("bth",),
    "nói chuyện": ("nc",),
    "mọi người": ("các fen",),
    "điện thoại": ("đt",),
    "tin nhắn": ("tn",),
}
_HELDOUT_WORDS = {
    "không": ("hổng",),
    "vào": ("dzô",),
    "về": ("dìa",),
    "thế": ("rứa",),
    "đâu": ("mô",),
    "bạn": ("fen",),
    "facebook": ("fb",),
}
_HELDOUT_LOOKALIKE = {"s": ("5",), "g": ("9",), "b": ("8",)}
_HELDOUT_SEPS = ("·",)
_QWERTY_NEIGHBOURS = {
    "a": "sqw", "b": "vghn", "c": "xdfv", "d": "serfcx", "e": "wsdr",
    "g": "ftyhbv", "h": "gyujnb", "i": "ujko", "k": "ijolm", "m": "njk",
    "n": "bhjm", "o": "iklp", "p": "ol", "q": "wa", "r": "edft",
    "s": "awedxz", "t": "rfgy", "u": "yhji", "v": "cfgb", "x": "zsdc", "y": "tghu",
}

def _eval_c1_heldout(text):
    rng = _sample_rng(78, text)
    out_lines = []
    for line in text.split("\n"):
        cur = line
        for src_p, alts in _HELDOUT_PHRASES.items():
            pat = re.compile(rf"(?<!\w){re.escape(src_p)}(?!\w)", re.IGNORECASE)
            if pat.search(cur):
                cur = pat.sub(lambda m: rng.choice(alts), cur, count=1)
        words_out = []
        mode = rng.choice(("lookalike", "sep", "typo"))
        edited = False
        for m in re.finditer(r"\S+|\s+", cur):
            tok_str = m.group(0)
            if tok_str.isspace():
                words_out.append(tok_str)
                continue
            lead, core, trail = _split_affix(tok_str)
            w = core
            if w and w.lower() in _HELDOUT_WORDS and rng.random() < 0.7:
                rep = rng.choice(_HELDOUT_WORDS[w.lower()])
                w = rep.capitalize() if w[0].isupper() else rep
            elif not edited and len(w) >= 4 and _ALPHA_CORE_RE.match(w):
                if mode == "lookalike":
                    ixs = [i for i in range(1, len(w)) if w[i].lower() in _HELDOUT_LOOKALIKE]
                    if ixs:
                        pick = rng.choice(ixs)
                        w = w[:pick] + rng.choice(_HELDOUT_LOOKALIKE[w[pick].lower()]) + w[pick + 1:]
                        edited = True
                elif mode == "sep":
                    cut = rng.randrange(1, len(w))
                    w = w[:cut] + rng.choice(_HELDOUT_SEPS) + w[cut:]
                    edited = True
                elif mode == "typo" and len(w) >= 5:
                    ixs = [i for i in range(1, len(w) - 1) if w[i] in _QWERTY_NEIGHBOURS]
                    if ixs:
                        pick = rng.choice(ixs)
                        w = w[:pick] + rng.choice(_QWERTY_NEIGHBOURS[w[pick]]) + w[pick + 1:]
                        edited = True
            words_out.append(lead + w + trail)
        out_lines.append("".join(words_out))
    return "\n".join(out_lines)

def _eval_blindspot_homoglyph(text):
    cmap = {"a": "а", "e": "е", "o": "о", "c": "с", "p": "р", "y": "у", "x": "х"}
    rng = _sample_rng(88, text)
    return "".join(cmap[ch] if ch in cmap and rng.random() < 0.40 else ch for ch in text)

def _eval_blindspot_zw(text):
    rng = _sample_rng(99, text)
    spans = [m.span() for m in re.finditer(r"[a-zA-ZÀ-ỹ]{3,}", text)]
    if not spans:
        return text
    picks = sorted(rng.sample(spans, min(3, len(spans))))
    out, prev = [], 0
    for sa, sb in picks:
        out.append(text[prev:sa])
        w = text[sa:sb]
        cut = rng.randrange(1, len(w))
        out.append(w[:cut] + "\u200b" + w[cut:])
        prev = sb
    out.append(text[prev:])
    return "".join(out)


# ==============================================================================
# 2. CHẠY STRESS TESTS (#5 THÔ) & XEM TRƯỚC CHUẨN HÓA (#6)
# ==============================================================================
stress_suites = {}
for name, fn, seen_flag in (
    ("accent_strip", strip_accents, ACCENT_AUG > 0 or SURFACE_AUG > 0),
    ("c1_in_dist", _eval_c1_in_dist, SURFACE_AUG > 0),
    ("c1_heldout_controlled", _eval_c1_heldout, False),
    ("blindspot_homoglyph", _eval_blindspot_homoglyph, False),
    ("blindspot_zero_width", _eval_blindspot_zw, False),
):
    res = _stress_eval(name, fn, seen_flag)
    if res is not None:
        stress_suites[name] = res
if "accent_strip" in stress_suites:
    s = stress_suites["accent_strip"]
    write_json(
        RUN / f"{report_key}_accent_stress.json",
        dict(
            n=s["n"],
            seen_in_train=s["seen_in_train"],
            original_f1=s["original_f1"],
            stripped_f1=s["perturbed_f1"],
            flip_rate=s["flip_rate"],
            newly_missed_harm=s["newly_missed_harm"],
        ),
    )
write_json(RUN / f"{report_key}_stress_tests.json", stress_suites)
print(pd.DataFrame(stress_suites).T)

if EVAL_WITH_NORMALIZER:
    norm_stress_suites = {}
    report_records_c0 = [e["row"] for e in report_encoded]
    raw_c0_eval = evaluate(final_pred, report_records_c0)
    norm_c0_pred = infer_normalized(model, report_records_c0, TEENCODE_DICT_MAP)
    norm_c0_eval = evaluate(norm_c0_pred, report_records_c0)
    norm_stress_suites["clean_c0"] = dict(
        suite="clean_c0",
        seen_in_train=True,
        n=len(norm_c0_pred),
        perturbed_raw_f1=float(raw_c0_eval["macro_f1"]),
        perturbed_norm_f1=float(norm_c0_eval["macro_f1"]),
        gain_vs_raw_f1=float(norm_c0_eval["macro_f1"] - raw_c0_eval["macro_f1"]),
        perturbed_norm_binary_f1=float(norm_c0_eval["binary_f1"]),
        fpr_an_toan=float(norm_c0_eval["fpr_an_toan"]),
        evidence_f1=float(norm_c0_eval["evidence"]["f1"]),
        evidence_f1_theo_y=float(norm_c0_eval["evidence"]["f1_theo_y"]),
        gain_vs_raw_ev_theo_y=float(norm_c0_eval["evidence"]["f1_theo_y"] - raw_c0_eval["evidence"]["f1_theo_y"]),
    )
    for name, fn, seen_flag in (
        ("accent_strip", strip_accents, ACCENT_AUG > 0 or SURFACE_AUG > 0),
        ("c1_in_dist", _eval_c1_in_dist, SURFACE_AUG > 0),
        ("c1_heldout_controlled", _eval_c1_heldout, False),
        ("blindspot_homoglyph", _eval_blindspot_homoglyph, False),
        ("blindspot_zero_width", _eval_blindspot_zw, False),
    ):
        changed_rows, orig_p = [], []
        for e, p in zip(report_encoded, final_pred):
            r = e["row"]
            v = fn(r["text"])
            if v != r["text"]:
                changed_rows.append(dict(r, text=v, bang_chung=[]))
                orig_p.append(p)
        if not changed_rows:
            continue
        sp = infer_normalized(model, changed_rows, TEENCODE_DICT_MAP)
        a_arr, y_arr = arrays(orig_p)
        b_arr, _ = arrays(sp)
        pf1 = float(macro(sp))
        raw_pf1 = stress_suites[name]["perturbed_f1"] if name in stress_suites else pf1
        norm_stress_suites[name] = dict(
            suite=name,
            seen_in_train=bool(seen_flag),
            n=len(sp),
            perturbed_raw_f1=raw_pf1,
            perturbed_norm_f1=pf1,
            gain_vs_raw_f1=pf1 - raw_pf1,
            perturbed_norm_binary_f1=float(f1_score(y_arr != 0, b_arr.argmax(1) != 0, zero_division=0)),
            newly_missed_harm=int(((y_arr != 0) & (a_arr.argmax(1) != 0) & (b_arr.argmax(1) == 0)).sum()),
        )
    write_json(RUN / f"{report_key}_norm_stress_tests.json", norm_stress_suites)
    print(f"=== Xem trước chuẩn hóa Dòng #6 trên {report_key} ===")
    print(pd.DataFrame(norm_stress_suites).T)


# ==============================================================================
# 3. STRESS TESTS BỔ SUNG TRÊN BỘ TEST THẬT (KHI ĐÃ MỞ KHÓA)
# ==============================================================================
if real_test_encoded is not None and real_test_pred is not None and RUN_REAL_TEST_PERTURB:
    rt_stress = {}
    for name, fn, seen_flag in (
        ("accent_strip", strip_accents, ACCENT_AUG > 0 or SURFACE_AUG > 0),
        ("c1_in_dist", _eval_c1_in_dist, SURFACE_AUG > 0),
        ("c1_heldout_controlled", _eval_c1_heldout, False),
        ("blindspot_homoglyph", _eval_blindspot_homoglyph, False),
        ("blindspot_zero_width", _eval_blindspot_zw, False),
    ):
        res = _stress_eval(name, fn, seen_flag, eval_encoded=real_test_encoded, eval_pred=real_test_pred)
        if res is not None:
            rt_stress[name] = res
    write_json(RUN / "real_test_stress_tests.json", rt_stress)
    print("=== Độ bền biến dạng bổ sung trên TEST THẬT (334 câu) ===")
    print(pd.DataFrame(rt_stress).T)


# ==============================================================================
# 4. MỐC ĐỐI CHIẾU ĐỘ DÀI (LENGTH BASELINE) & PHÉP THỬ BỎ NGỮ CẢNH ĐA LƯỢT
# ==============================================================================
# Baseline chỉ dùng độ dài. Mô hình này không đọc nội dung hoặc metadata nguồn.
from sklearn.tree import DecisionTreeClassifier
def lengths(e):
    t = e["row"]["text"]
    return [len(t.splitlines()), len(t), len(t.split())]
length_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, class_weight="balanced", random_state=SEED)
length_model.fit([lengths(e) for e in encoded["fit"]], [e["label"] for e in encoded["fit"]])
comparison_data = report_encoded
length_pred = length_model.predict([lengths(e) for e in comparison_data])
diag = {"length_only_macro_f1": float(f1_score([e["label"] for e in comparison_data], length_pred, average="macro", labels=range(5), zero_division=0))}
# So trên đúng các mẫu đa lượt và cùng nhãn đoạn. Đây là phép bỏ ngữ cảnh tại inference.
mult = [(e, p) for e, p in zip(comparison_data, final_pred) if e["row"]["so_dong"] > 1]
if mult:
    last_turn = [prepare(dict(e["row"], text=e["row"]["text"].split("\n")[-1], bang_chung=[])) for e, _ in mult]
    truncated_pred = infer(model, last_turn)
    diag.update(
        multi_n=len(mult),
        full_context_multi_macro_f1=float(macro([p for _, p in mult])),
        last_turn_multi_macro_f1=float(macro(truncated_pred)),
    )
write_json(RUN / f"{report_key}_context_checks.json", diag)
print(diag)


# ==============================================================================
# 5. BENCHMARK VRAM & ĐỘ TRỄ SUY LUẬN SẠCH (P50 / P95)
# ==============================================================================
@torch.inference_mode()
def benchmark(net, data, device, limit=100):
    net.eval()
    net.zero_grad(set_to_none=True)
    gc.collect()
    samples = data[:limit]
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    def call(e):
        with (amp() if device.type == "cuda" else nullcontext()):
            return net(to_device(collate([e]), device))
    if device.type == "cuda":
        torch.cuda.empty_cache()
        sync()
        weights_mib = (
            sum(p.numel() * p.element_size() for p in net.parameters() if p.is_cuda)
            + sum(b.numel() * b.element_size() for b in net.buffers() if b.is_cuda)
        ) / 2**20
        baseline_mib = torch.cuda.memory_allocated(device) / 2**20
        if baseline_mib > weights_mib * 1.35 + 64.0:
            print(f"CẢNH BÁO VRAM: baseline ({baseline_mib:.1f} MiB) lớn hơn đáng kể trọng số mô hình ({weights_mib:.1f} MiB)!")
    else:
        weights_mib = baseline_mib = None
    for e in samples[:5]:
        call(e)
    sync()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    latency, prep = [], []
    for e in samples:
        start = time.perf_counter()
        prepare(e["row"])
        prep.append(1000 * (time.perf_counter() - start))
        start = time.perf_counter()
        call(e)
        sync()
        latency.append(1000 * (time.perf_counter() - start))
    peak_mib = torch.cuda.max_memory_allocated(device) / 2**20 if device.type == "cuda" else None
    return dict(
        n=len(samples),
        batch=1,
        model_and_collate_p50_ms=float(np.median(latency)),
        model_and_collate_p95_ms=float(np.percentile(latency, 95)),
        prepare_p50_ms=float(np.median(prep)),
        model_weights_cuda_MiB=(round(weights_mib, 2) if weights_mib is not None else None),
        inference_baseline_cuda_MiB=(round(baseline_mib, 2) if baseline_mib is not None else None),
        session_peak_cuda_MiB=(round(peak_mib, 2) if peak_mib is not None else None),
        activation_overhead_cuda_MiB=(round(max(0.0, peak_mib - baseline_mib), 2) if peak_mib is not None else None),
    )
perf = benchmark(model, report_encoded, DEVICE)
write_json(RUN / "benchmark.json", perf)
print(perf)


# ==============================================================================
# 6. HÀM SUY LUẬN MẪU PREDICT_TEXT() & CHẠY THỬ
# ==============================================================================
def predict_text(net, text, device=DEVICE, use_normalizer=False):
    if not text.strip() or not 1 <= len(text.split("\n")) <= 8:
        raise ValueError("Nhập 1–8 lượt, mỗi lượt một dòng")
    stub = dict(id="demo", text=text, nhan="AN_TOAN", bang_chung=[])
    if use_normalizer:
        tmap = TEENCODE_DICT_MAP or fetch_teencode_dict()
        item = infer_normalized(net, [stub], tmap, infer_fn=lambda m, d: infer(m, d, device=device))[0]
    else:
        item = infer(net, [prepare(stub)], device=device)[0]
    return format_output(item, text)
restore(model, load_file(str(RUN / "adapter.safetensors")))
print(predict_text(model, "Chiều nay nhóm mình họp lúc mấy giờ?\nBa giờ ở thư viện nhé."))

---
<a id="sec-8"></a>
## 8. Lượng tử hóa INT8 trên CPU (bao gồm `embed_tokens`)

Ô tùy chọn phục vụ đóng gói và đánh giá hiệu năng trên CPU thiết bị đích:

| Cờ điều khiển | Cơ chế thực thi & Chỉ số đối chiếu |
| :--- | :--- |
| **`RUN_INT8_CPU = True`** | Gộp toàn bộ LoRA vào các lớp tuyến tính, nén bảng `embed_tokens` bằng `PerRowInt8Embedding` (INT8 đối xứng theo hàng + scale FP32: `593,5 MiB → 148,7 MiB`) khi `QUANTIZE_EMBEDDINGS = True`, và lượng tử hóa động các lớp `nn.Linear` của backbone sang `qint8`. So sánh FP32 vs INT8 trên cùng mẫu, cùng CPU, cùng ngưỡng; ghi nhận cả `fp32_serialized_MiB` lẫn `serialized_MiB` trong `int8_comparison.json`. |

In [ ]:
#@title 8.1. Gộp LoRA & Lượng tử hóa INT8 trên CPU (PerRowInt8Embedding + Linear)
# ==============================================================================
# 1. LỚP PER-ROW INT8 EMBEDDING (NÉN 4×, SCALE FP32)
# ==============================================================================
class PerRowInt8Embedding(nn.Module):
    """Lượng tử hóa đối xứng INT8 theo từng hàng từ vựng (nén ~4x, giải nén cục bộ chỉ trên token đang tra cứu)."""
    def __init__(self, emb: nn.Embedding):
        super().__init__()
        w = emb.weight.detach().float().cpu()
        self.num_embeddings = emb.num_embeddings
        self.embedding_dim = emb.embedding_dim
        self.padding_idx = emb.padding_idx
        scale = w.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / 127.0
        qweight = (w / scale).round().clamp(-127, 127).to(torch.int8)
        if self.padding_idx is not None:
            qweight[self.padding_idx].zero_()
            scale[self.padding_idx].zero_()
        self.register_buffer("qweight", qweight)
        self.register_buffer("scale", scale.to(torch.float32))

    def forward(self, input_ids):
        q = F.embedding(input_ids, self.qweight).float()
        s = F.embedding(input_ids, self.scale)
        return q * s

def quantize_embeddings_int8(module, min_num_embeddings=4096):
    replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Embedding) and child.num_embeddings >= min_num_embeddings:
            setattr(module, name, PerRowInt8Embedding(child))
            replaced += 1
        else:
            replaced += quantize_embeddings_int8(child, min_num_embeddings=min_num_embeddings)
    return replaced


# ==============================================================================
# 2. GỘP LORA & ĐÁNH GIÁ NÉN INT8 TRÊN CPU (DUNG LƯỢNG, ĐỘ TRỄ & F1)
# ==============================================================================
if RUN_INT8_CPU:
    import copy, io
    cpu = torch.device("cpu")
    if "x86" in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = "x86"
    subset = report_encoded[:min(256, len(report_encoded))]
    # Kết thúc huấn luyện rồi mới đưa backbone về CPU và gộp LoRA (attention + MLP).
    cpu_net = copy.deepcopy(model.to(cpu)).float().eval()
    model.to(DEVICE)
    cpu_net.backbone.gradient_checkpointing_disable()
    for layer in cpu_net.backbone.layers:
        modules = [(layer.self_attn, ("q_proj", "k_proj", "v_proj", "o_proj"))]
        if LORA_MLP:
            modules.append((layer.mlp, ("gate_proj", "up_proj", "down_proj")))
        for parent, names in modules:
            for name in names:
                adapter = getattr(parent, name)
                with torch.no_grad():
                    adapter.base.weight.addmm_(adapter.up.weight, adapter.down.weight, alpha=adapter.scale)
                setattr(parent, name, adapter.base)
    fp_buf = io.BytesIO()
    torch.save(cpu_net.state_dict(), fp_buf)
    fp32_mib = len(fp_buf.getvalue()) / 2**20
    del fp_buf
    fp = infer(cpu_net, subset, device=cpu)
    fp_perf = benchmark(cpu_net, subset, cpu)
    emb_replaced = quantize_embeddings_int8(cpu_net.backbone, min_num_embeddings=4096) if QUANTIZE_EMBEDDINGS else 0
    from torch.ao.quantization import quantize_dynamic
    quantize_dynamic(cpu_net.backbone, {nn.Linear}, dtype=torch.qint8, inplace=True)
    qi = infer(cpu_net, subset, device=cpu)
    tmp = RUN / "int8.tmp"
    torch.save(cpu_net.state_dict(), tmp)
    os.replace(tmp, RUN / "int8_cpu.pt")
    quant_result = dict(
        n=len(subset),
        engine=torch.backends.quantized.engine,
        quantize_embeddings=QUANTIZE_EMBEDDINGS,
        embeddings_quantized=emb_replaced,
        fp32_serialized_MiB=round(fp32_mib, 2),
        serialized_MiB=round((RUN / "int8_cpu.pt").stat().st_size / 2**20, 2),
        fp32=evaluate(fp, [e["row"] for e in subset]),
        int8=evaluate(qi, [e["row"] for e in subset]),
        fp32_perf=fp_perf,
        int8_perf=benchmark(cpu_net, subset, cpu),
    )
    write_json(RUN / "int8_comparison.json", quant_result)
    print(
        f"FP32 / INT8 macro-F1: {macro(fp):.4f} / {macro(qi):.4f} | "
        f"Size: {quant_result['fp32_serialized_MiB']:.1f} -> {quant_result['serialized_MiB']:.1f} MiB (emb_q={emb_replaced})"
    )
    del cpu_net
    gc.collect()

---
<a id="sec-9"></a>
## 9. Quy trình thực nghiệm chuẩn, Ablation & Đánh giá Bộ Test khóa

1. **Kiểm thử & So sánh ứng viên song song (`DATA_SCHEME = "dev_80_10_10"`, `EVALUATE_REAL_TEST = False`):**
   - Chạy `RUN_MODE = "smoke"` để xác minh GPU, tải trọng số, kiểm định padding và cơ chế lưu checkpoint; sau đó đổi `RUN_MODE = "full"`.
   - Toàn bộ tệp kết quả (`adapter.safetensors`, `config.json`, `split_ids.json`, `history.json`, `calibration.json`, `dev_metrics.json`, `dev_predictions.jsonl`, `dev_norm_stress_tests.json`, `benchmark.json`) khớp 100% ID `dev`/`cal` với Nhánh 2 ([`v2_dualorc_crf_colab.ipynb`](v2_dualorc_crf_colab.ipynb)).
2. **Ma trận Ablation (giữ cùng `SEED = 42` và `DATA_SCHEME`):**
   - Trộn trọng số: `MERGE_RATIO = 0.0` vs `0.5`.
   - Phân cấp hội thoại: `TURN_MODE = "flat"` vs `"hier"` · `HEAD_ARCH = "v3"` vs `"v4"`.
   - Đa nhiệm & Bền vững: `SPAN_WEIGHT = 0.0` vs `0.35` · `CONSISTENCY_WEIGHT = 0.0` vs `0.25` · tắt tăng cường `ACCENT_AUG = 0.0, SURFACE_AUG = 0.0`.
3. **Huấn luyện trên tập dữ liệu mở rộng (tùy chọn sau khi xác định kiến trúc & `EPOCHS` trên `dev`):**
   - Chuyển `DATA_SCHEME = "final_90_10"` (`fit = 12.053`, `cal = 1.339`) hoặc `DATA_SCHEME = "final_100_human_cal"` (`fit = 13.392` + `HUMAN_CAL_PATH` do người gán), huấn luyện đủ `EPOCHS` cố định rồi hiệu chuẩn và đóng băng `calibration.json`.
4. **Đánh giá một lần duy nhất trên Bộ Test khóa (`334 C0 + 668 C1 + 668 C2 = 1.670` mẫu):**
   - Sau khi hoàn tất lựa chọn cấu hình và đóng băng `calibration.json`, bật `EVALUATE_REAL_TEST = True` (không đổi `RUN` hash, không ghi đè `calibration.json`) để chấm **đúng 1 lần duy nhất** cả Dòng `#5` thô lẫn Dòng `#6` chuẩn hóa đầu vào (`EVAL_WITH_NORMALIZER = True`).
   - Hệ thống tự khóa qua `REAL_TEST_LOCK.json` và xuất `real_test_all_predictions.jsonl` / `real_test_norm_all_predictions.jsonl` tương thích 100% với bộ chấm CLI chuẩn của kho mã nguồn:
     ```bash
     python -m src.v2.eval.evaluate \
       --gold data/v2/test_day_du.jsonl data/v2/test_perturb_C1.jsonl data/v2/test_perturb_C2.jsonl \
       --pred <RUN>/real_test_all_predictions.jsonl \
       --meta data/v2/test_day_du.jsonl \
       --out results/v2/<ten_cau_hinh>/metrics.json
     ```
5. **Đo chi phí & Lượng tử hóa:** Khi bật `RUN_INT8_CPU = True`, đối chiếu lại F1 nhãn hiếm, bằng chứng (mức từ & F1 theo ý), độ trễ p50/p95 và dung lượng sau khi nén INT8 cả `embed_tokens` lẫn `nn.Linear`.

---
<a id="sec-10"></a>
## 10. Đối chiếu API & Điều kiện kiểm chứng thực nghiệm

| Phần | Cách áp dụng trong notebook | Tài liệu tham chiếu |
| :--- | :--- | :--- |
| **Cài đặt và Drive** | Kiểm tra phiên bản đang nạp trong kernel, ghi `environment.json` ngay khi tạo thư mục `RUN`. Dữ liệu và model tải về ổ runtime, checkpoint ghi vào Drive theo epoch. | [Colab FAQ](https://research.google.com/colaboratory/faq.html) · [Transformers 4.57.6](https://pypi.org/project/transformers/4.57.6/) |
| **Tải và trộn trọng số** | Ghim revision, kiểm tra HTTP retry và SHA-256 của JSONL. Đọc tensor SafeTensors theo tên, kiểm tra hình dạng và chỉ trộn các hàng embedding có chung token. | [Requests](https://requests.readthedocs.io/en/latest/user/quickstart/#timeouts) · [HF Hub](https://huggingface.co/docs/huggingface_hub/guides/download) · [SafeTensors](https://huggingface.co/docs/safetensors/api/torch) |
| **Chia tập và bằng chứng** | `StratifiedGroupKFold` giữ nguyên nhóm. Kiểm tra offset trên hai lượt với NFC, NFD và bỏ dấu. Khi ghép chunk, offset vẫn thuộc văn bản gốc. | [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html) · [Tokenizers](https://huggingface.co/docs/tokenizers/python/latest/quicktour.html) |
| **Qwen3 và Padding** | Dùng đường truyền mặt nạ `full_attention` của phiên bản đã ghim. SDPA nhận `True` tại vị trí được nhìn. Ô khởi tạo so logits và điểm token khi thêm padding, đồng thời thử đổi token phía sau để xác minh tính hai chiều. | [Mã Qwen3 v4.57.6](https://github.com/huggingface/transformers/blob/v4.57.6/src/transformers/models/qwen3/modeling_qwen3.py) · [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) |
| **Huấn luyện** | BF16 cần hỗ trợ phần cứng; FP16 dùng `GradScaler`. Cross-entropy nhãn và nguyên mẫu dùng cùng mẫu số tổng trọng số lớp; BCE dùng số token có nhãn. Mỗi cửa sổ tích lũy chỉ unscale, clip và step một lần. | [BF16](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_bf16_supported.html) · [AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) · [CE](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) · [BCE](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) |
| **Lưu và tiếp tục** | Activation checkpointing dùng `use_reentrant=False`. Tệp `last.pt` giữ trọng số đang học, bản tốt nhất và trạng thái AdamW, lịch LR, scaler. Mỗi epoch có seed xác định theo chỉ số epoch; khôi phục yêu cầu khớp cấu hình và dtype. | [Activation Checkpoint](https://docs.pytorch.org/docs/stable/checkpoint.html) · [Lưu/Nạp Checkpoint](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html) · [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) |
| **Hiệu chuẩn và báo cáo** | Kiểm tra hội tụ của SciPy trước khi lưu nhiệt độ. Chọn ngưỡng trên `cal`, tính macro-F1 và báo khoảng bootstrap nhóm 95% CI trên tập báo cáo. Bảng tần suất và biểu đồ dùng pandas và Matplotlib. | [SciPy Optimize](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize_scalar.html) · [F1 Score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) · [NumPy Quantile](https://numpy.org/doc/stable/reference/generated/numpy.quantile.html) · [pandas](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [Matplotlib](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.savefig.html) |
| **Benchmark & INT8** | Đồng bộ CUDA trước khi đọc thời gian. Số đo VRAM là đỉnh tensor được PyTorch cấp trong phiên benchmark. INT8 động là thử nghiệm CPU tùy chọn qua API eager, kết quả phụ thuộc engine và bản PyTorch. | [CUDA Memory](https://docs.pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html) · [Quantization](https://docs.pytorch.org/docs/stable/quantization.html) |

> **Xác nhận môi trường thực tế:** Chạy `RUN_MODE = "smoke"` trên GPU, kiểm tra `environment.json`, sau đó mở phiên mới có cùng dtype và chạy lại cùng cấu hình để xác nhận ô huấn luyện khôi phục đúng epoch đã lưu. Drive và RAM/VRAM trên thiết bị đích cần kiểm chứng trực tiếp.

---
<a id="sec-11"></a>
## 11. Phụ lục: Tương thích thư viện & Đặc tả quy ước đồng bộ

### 11.1. Bộ thư viện ghim

| Gói | Phiên bản ghim | Vai trò |
| :--- | :---: | :--- |
| `transformers` | `4.57.6` | Backbone `Qwen3Model` với mặt nạ `full_attention` tường minh |
| `tokenizers` | `0.22.2` | Fast tokenizer trả về `offsets_mapping` chính xác về chuỗi gốc |
| `safetensors` | `0.6.2` | Nạp/lưu trọng số an toàn và ánh xạ tensor trộn theo tên |
| `huggingface-hub` | `0.36.0` | Tải tệp mô hình ghim chặt theo revision SHA |
| `scikit-learn` | `1.9.1` | `StratifiedGroupKFold`, tính F1, confusion matrix và log-loss |

### 11.2. Đặc tả quy ước thực nghiệm đồng bộ liên nhánh

Toàn bộ giao thức dữ liệu, tối ưu bền vững và đánh giá được thống nhất với Nhánh 2 ([`v2_dualorc_crf_colab.ipynb`](v2_dualorc_crf_colab.ipynb)) nhằm bảo đảm so sánh công bằng:

| Hạng mục | Quy ước áp dụng |
| :--- | :--- |
| **1. Toàn vẹn dữ liệu & Khóa test** | Ghim commit `DATA_REV = 371a0f1caf3f7b62f62b87dbf557c66eda9c544a` cùng 6 mã SHA-256 (`so_sanh`, `train`, `test` 334, `test_day_du` 334, `test_perturb_C1` 668, `test_perturb_C2` 668) và chặn cả 6 tệp tại khóa an toàn `HUMAN_CAL_PATH`. |
| **2. Tăng cường phi điểm mù & Dual-View JS** | Ghép cặp `(gốc, bỏ dấu/C1)` trong `collate`/`split_microbatch` với `CONSISTENCY_WEIGHT = 0.25`, `ACCENT_AUG = 0.30`, `SURFACE_AUG = 0.20` (từ `>= 3` ký tự, ngân sách theo độ dài dòng, giữ hoàn toàn C2 và C1 held-out ngoài train) và `LABEL_SMOOTHING = 0.02`. |
| **3. Chọn checkpoint & Hiệu chuẩn xác suất** | Tiêu chí `checkpoint_select_key` (`real_aligned`) dùng `0.50 * macro_f1 + 0.15 * that_macro_f1 + 0.25 * that_acc + 0.10 * ev_f1` (`0.55/0.20/0.25` khi `SPAN_WEIGHT == 0`, tương ứng tỷ lệ `0.50/0.15/0.25/0.10`); hiệu chuẩn nhiệt độ dùng trần trọng số `cap = 3.0`, biên `T >= 0.74` trên `internal_llm`; đầu ra xuất đủ `probs` 5 lớp và `worst_group_region_macro_f1`. |
| **4. Đánh giá song song `#5` & `#6`** | Xuất đồng thời `real_test_all_predictions.jsonl` (thô `#5`) và `real_test_norm_all_predictions.jsonl` (+ chuẩn hóa giữ dòng `\n`, `#6`, 1.670 dòng) tương thích 100% `python -m src.v2.eval.evaluate` kèm ma trận 15 lát cắt `real_test_official_matrix.json`. |